### Dataset Loading and Environment Configuration
Objective: Load and validate the improved Phase 3 dataset before developing the new Transformer architectures.
The V3 dataset contains harmonically filtered training windows, preserved chord annotations, recovered high-confidence chord labels, and event-aware harmonic-change supervision.
Only training and validation records are loaded for model development. The test partition remains reserved.
All experiments will use the same musical input representation and vocabulary as the original Phase 3 Transformers to support meaningful comparisons.

In [1]:
# ==========================================
# PHASE 3 — IMPROVED TRANSFORMER DEVELOPMENT
# CELL 1 — DATASET LOADING AND VERIFICATION
# ==========================================

import json
import re
import random

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch


# ==========================================
# 1. Reproducibility
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ==========================================
# 2. Project and dataset paths
# ==========================================

PROJECT_ROOT = (
    Path.home()
    / "Developer"
    / "vocal-harmonization"
)

DATASET_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "mtg-jamendo"
    / "phase3-prepared"
    / "windows_v3_filtered_event_aware.jsonl"
)

assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}"
)


# ==========================================
# 3. Define label vocabularies
# ==========================================

ROOT_LABELS = [
    "I", "♭II", "II", "♭III",
    "III", "IV", "♯IV", "V",
    "♭VI", "VI", "♭VII", "VII"
]

FAMILY_LABELS = [
    "MAJ",
    "MIN",
    "DOM",
    "DIM",
    "HDIM",
    "AUG",
    "SUS"
]

GENRE_LABELS = [
    "rock",
    "pop",
    "electronic",
    "folk",
    "jazz"
]

NUM_ROOTS = len(ROOT_LABELS)
NUM_FAMILIES = len(FAMILY_LABELS)

IGNORE_INDEX = -100


# ==========================================
# 4. Load train and validation records
# ==========================================

# The test partition is deliberately
# excluded from model development.

examples = []

split_counts = Counter()

split_pattern = re.compile(
    r'"split"\s*:\s*"(train|val|test)"'
)

with open(
    DATASET_PATH,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        # Inspect only split metadata first.
        # Do not parse test record contents.

        match = split_pattern.search(line)

        if match is None:
            raise ValueError(
                "Record has no recognized split."
            )

        split = match.group(1)

        if split == "test":
            continue

        record = json.loads(line)

        examples.append(record)
        split_counts[split] += 1


# ==========================================
# 5. Verify dataset dimensions
# ==========================================

assert split_counts["train"] == 4523
assert split_counts["val"] == 1405

assert len(examples) == 5928

train_examples = [
    ex for ex in examples
    if ex["split"] == "train"
]

val_examples = [
    ex for ex in examples
    if ex["split"] == "val"
]


print("PHASE 3 — IMPROVED TRANSFORMER DATASET")
print("--------------------------------")

print("Dataset:", DATASET_PATH.name)

print("Training windows:", len(train_examples))
print("Validation windows:", len(val_examples))

print("Test records loaded: 0")


# ==========================================
# 6. Check available songs and artists
# ==========================================

def count_entities(records, field):

    return len({
        ex[field]
        for ex in records
        if ex.get(field) is not None
    })


print("\nSONG AND ARTIST COVERAGE")
print("--------------------------------")

print(
    "Training songs:",
    count_entities(train_examples, "track_id")
)

print(
    "Validation songs:",
    count_entities(val_examples, "track_id")
)

print(
    "Training artists:",
    count_entities(train_examples, "artist_id")
)

print(
    "Validation artists:",
    count_entities(val_examples, "artist_id")
)


# ==========================================
# 7. Verify V3 supervision
# ==========================================

statistics = {
    "train": Counter(),
    "val": Counter()
}

for ex in examples:

    split = ex["split"]

    c = statistics[split]

    targets = ex["v3_targets"]

    old_targets = ex["beat_targets"]

    n_beats = len(ex["beat_grid"])

    # Every supervision sequence must have
    # exactly one entry per beat.

    assert len(old_targets) == n_beats

    assert all(
        len(values) == n_beats
        for values in targets.values()
    )

    assert len(ex["genre_vector"]) == 5

    c["windows"] += 1
    c["beats"] += n_beats

    for i, old in enumerate(old_targets):

        root = targets["root_ids"][i]
        family = targets["family_ids"][i]

        chord_mask = targets["chord_mask"][i]

        source = targets[
            "chord_label_source"
        ][i]

        change = targets["change_labels"][i]

        change_mask = targets["change_mask"][i]

        old_valid = (
            old.get("model_target_mask") == 1
            and old.get("root_id") is not None
            and old.get("family_id") is not None
        )

        # ----------------------------------
        # A. Verify original labels
        # ----------------------------------

        if old_valid:

            assert chord_mask == 1
            assert source == 1

            assert root == int(old["root_id"])
            assert family == int(old["family_id"])

            c["original_labels"] += 1

        # ----------------------------------
        # B. Verify new chord targets
        # ----------------------------------

        if chord_mask == 1:

            assert 0 <= root < NUM_ROOTS
            assert 0 <= family < NUM_FAMILIES

            c["chord_targets"] += 1

            if source == 2:

                assert split == "train"
                assert not old_valid

                c["recovered_labels"] += 1

        else:

            assert root == IGNORE_INDEX
            assert family == IGNORE_INDEX
            assert source == 0

        # ----------------------------------
        # C. Verify change targets
        # ----------------------------------

        if change_mask == 1:

            assert change in (0, 1)

            c["change_targets"] += 1

            if change == 1:
                c["change_positive"] += 1

            else:
                c["change_negative"] += 1

        else:

            assert change == IGNORE_INDEX

            c["change_masked"] += 1


# ==========================================
# 8. Supervision summary
# ==========================================

summary_rows = []

for split in ["train", "val"]:

    c = statistics[split]

    summary_rows.append({

        "Split": split,

        "Windows": c["windows"],

        "Total beats": c["beats"],

        "Chord targets": c["chord_targets"],

        "Recovered chords": c["recovered_labels"],

        "Change targets": c["change_targets"],

        "Change positives": c["change_positive"],

        "Change negatives": c["change_negative"],

        "Change coverage":
            c["change_targets"] / c["beats"]
    })


summary_df = pd.DataFrame(summary_rows)

print("\nV3 SUPERVISION SUMMARY")
print("--------------------------------")

display(summary_df.round(4))


# ==========================================
# 9. Verify expected supervision counts
# ==========================================

assert (
    statistics["train"]["chord_targets"]
    == 239822
)

assert (
    statistics["train"]["recovered_labels"]
    == 220
)

assert (
    statistics["train"]["change_targets"]
    == 253222
)

assert (
    statistics["train"]["change_positive"]
    == 54422
)

assert (
    statistics["val"]["chord_targets"]
    == 77718
)

assert (
    statistics["val"]["recovered_labels"]
    == 0
)

assert (
    statistics["val"]["change_positive"]
    == 14615
)


# ==========================================
# 10. Check artist-disjoint splitting
# ==========================================

train_artists = {
    ex["artist_id"]
    for ex in train_examples
    if ex.get("artist_id") is not None
}

val_artists = {
    ex["artist_id"]
    for ex in val_examples
    if ex.get("artist_id") is not None
}

assert train_artists.isdisjoint(val_artists)

print("\nDATASET SPLIT VERIFICATION")
print("--------------------------------")

print("Training/validation artists overlap: NO")


# ==========================================
# 11. Inspect V3 target structure
# ==========================================

sample = train_examples[0]

print("\nSAMPLE WINDOW")
print("--------------------------------")

print("Track:", sample["track_id"])

print("Window:", sample["window_index"])

print("Melody notes:", len(sample["melody"]))

print("Beat positions:", len(sample["beat_grid"]))

print("V3 target fields:")

for key in sample["v3_targets"]:
    print(" -", key)


# ==========================================
# 12. Final verification
# ==========================================

print("\nFINAL VERIFICATION")
print("--------------------------------")

print("V3 dataset loaded: YES")
print("Original chord targets preserved: YES")

print("Recovered chord targets available: YES")
print("Event-aware change targets available: YES")

print("Training and validation only: YES")
print("Test labels accessed: NO")

print("Dataset modified: NO")
print("Model training performed: NO")

print("\nReady for PyTorch Dataset construction!")

PHASE 3 — IMPROVED TRANSFORMER DATASET
--------------------------------
Dataset: windows_v3_filtered_event_aware.jsonl
Training windows: 4523
Validation windows: 1405
Test records loaded: 0

SONG AND ARTIST COVERAGE
--------------------------------
Training songs: 701
Validation songs: 176
Training artists: 148
Validation artists: 33

V3 SUPERVISION SUMMARY
--------------------------------


,Split,Windows,Total beats,Chord targets,Recovered chords,Change targets,Change positives,Change negatives,Change coverage
0,train,4523,272175,239822,220,253222,54422,198800,0.9304
1,val,1405,86624,77718,0,81859,14615,67244,0.9450



DATASET SPLIT VERIFICATION
--------------------------------
Training/validation artists overlap: NO

SAMPLE WINDOW
--------------------------------
Track: track_0000382
Window: 0
Melody notes: 63
Beat positions: 64
V3 target fields:
 - root_ids
 - family_ids
 - chord_mask
 - change_labels
 - change_mask
 - chord_occupancy
 - mapped_coverage
 - chord_label_source

FINAL VERIFICATION
--------------------------------
V3 dataset loaded: YES
Original chord targets preserved: YES
Recovered chord targets available: YES
Event-aware change targets available: YES
Training and validation only: YES
Test labels accessed: NO
Dataset modified: NO
Model training performed: NO

Ready for PyTorch Dataset construction!


In [2]:
# ============================================================
# MODEL 1 — IMPROVED TEMPORAL TRANSFORMER
# Complete training experiment
# ============================================================

import math
import time
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

BATCH_SIZE = 16
MAX_EPOCHS = 20
PATIENCE = 5

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

D_MODEL = 128
NHEAD = 4
ENCODER_LAYERS = 2
DECODER_LAYERS = 2
DIM_FEEDFORWARD = 512
DROPOUT = 0.1

TEMPORAL_SIGMA = 2.0
TEMPORAL_STRENGTH = 1.0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(SEED)

EXPERIMENT_NAME = "phase3_v3_temporal_weighted"

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "pasan"
    / "checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / f"{EXPERIMENT_NAME}_best.pt"
)


# ============================================================
# 2. V3 Dataset
# ============================================================

class V3HarmonyDataset(Dataset):

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        ex = self.records[idx]

        duration = max(
            float(ex["duration_sec"]),
            1e-6
        )

        melody = ex["melody"]
        beats = ex["beat_grid"]
        targets = ex["v3_targets"]

        return {
            "melody_pitch": torch.tensor(
                [int(n["midi_pitch"]) for n in melody],
                dtype=torch.long
            ),

            "melody_time": torch.tensor(
                [
                    [
                        n["onset_sec"] / duration,
                        n["duration_sec"] / duration
                    ]
                    for n in melody
                ],
                dtype=torch.float32
            ).reshape(-1, 2),

            "beat_time": torch.tensor(
                [
                    [
                        b["time_sec"] / duration,
                        b["duration_sec"] / duration
                    ]
                    for b in beats
                ],
                dtype=torch.float32
            ),

            "bar_index": torch.tensor(
                [b["bar_index"] for b in beats],
                dtype=torch.long
            ),

            "beat_position": torch.tensor(
                [b["beat_in_bar"] for b in beats],
                dtype=torch.long
            ),

            "meter": torch.tensor(
                [b["meter_numerator"] for b in beats],
                dtype=torch.long
            ),

            "target_root": torch.tensor(
                targets["root_ids"],
                dtype=torch.long
            ),

            "target_family": torch.tensor(
                targets["family_ids"],
                dtype=torch.long
            ),

            "change_labels": torch.tensor(
                targets["change_labels"],
                dtype=torch.long
            ),

            "track_id": ex["track_id"]
        }


def v3_collate(batch):

    padding = {
        "melody_pitch": 0,
        "melody_time": 0.0,
        "beat_time": 0.0,
        "bar_index": 0,
        "beat_position": 0,
        "meter": 0,
        "target_root": -100,
        "target_family": -100,
        "change_labels": -100
    }

    result = {}

    for key, value in padding.items():

        result[key] = pad_sequence(
            [item[key] for item in batch],
            batch_first=True,
            padding_value=value
        )

    melody_lengths = torch.tensor([
        len(item["melody_pitch"])
        for item in batch
    ])

    beat_lengths = torch.tensor([
        len(item["target_root"])
        for item in batch
    ])

    result["melody_padding_mask"] = (
        torch.arange(
            result["melody_pitch"].shape[1]
        )[None, :] >= melody_lengths[:, None]
    )

    result["beat_padding_mask"] = (
        torch.arange(
            result["target_root"].shape[1]
        )[None, :] >= beat_lengths[:, None]
    )

    result["track_ids"] = [
        item["track_id"] for item in batch
    ]

    return result


train_loader = DataLoader(
    V3HarmonyDataset(train_examples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=v3_collate
)

val_loader = DataLoader(
    V3HarmonyDataset(val_examples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=v3_collate
)


# ============================================================
# 3. Positional Encoding
# ============================================================

def add_positional_encoding(x):

    _, length, dimension = x.shape

    positions = torch.arange(
        length,
        device=x.device,
        dtype=x.dtype
    ).unsqueeze(1)

    frequencies = torch.exp(
        torch.arange(
            0, dimension, 2,
            device=x.device,
            dtype=x.dtype
        ) * (-math.log(10000.0) / dimension)
    )

    angles = positions * frequencies

    pe = torch.zeros(
        length, dimension,
        device=x.device,
        dtype=x.dtype
    )

    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)

    return x + pe.unsqueeze(0)


# ============================================================
# 4. Gaussian Temporal Attention Bias
# ============================================================

def build_temporal_bias(batch):

    beat_start = (
        batch["beat_time"][:, :, 0].unsqueeze(-1)
    )

    beat_duration = (
        batch["beat_time"][:, :, 1].unsqueeze(-1)
    )

    beat_end = beat_start + beat_duration

    note_start = (
        batch["melody_time"][:, :, 0].unsqueeze(1)
    )

    note_duration = (
        batch["melody_time"][:, :, 1].unsqueeze(1)
    )

    note_end = note_start + note_duration

    gap = torch.maximum(
        note_start - beat_end,
        beat_start - note_end
    ).clamp(min=0)

    distance = (
        gap / beat_duration.clamp(min=1e-6)
    )

    bias = (
        -0.5
        * TEMPORAL_STRENGTH
        * (distance / TEMPORAL_SIGMA).square()
    )

    bias = bias.clamp(min=-20)

    bias = bias.masked_fill(
        batch["melody_padding_mask"].unsqueeze(1),
        float("-inf")
    )

    B, T, S = bias.shape

    return (
        bias.unsqueeze(1)
        .expand(B, NHEAD, T, S)
        .reshape(B * NHEAD, T, S)
    )


# ============================================================
# 5. Original Temporal Transformer Architecture
# ============================================================

class HarmonyTemporalTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.pitch_embedding = nn.Embedding(
            128, D_MODEL
        )

        self.melody_time_projection = nn.Linear(
            2, D_MODEL
        )

        self.melody_norm = nn.LayerNorm(D_MODEL)

        self.beat_time_projection = nn.Linear(
            2, D_MODEL
        )

        self.bar_embedding = nn.Embedding(
            16, D_MODEL
        )

        self.beat_position_embedding = nn.Embedding(
            10, D_MODEL,
            padding_idx=0
        )

        self.meter_embedding = nn.Embedding(
            10, D_MODEL,
            padding_idx=0
        )

        self.beat_norm = nn.LayerNorm(D_MODEL)

        self.dropout = nn.Dropout(DROPOUT)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=ENCODER_LAYERS,
            norm=nn.LayerNorm(D_MODEL),
            enable_nested_tensor=False
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=DECODER_LAYERS,
            norm=nn.LayerNorm(D_MODEL)
        )

        self.root_head = nn.Linear(
            D_MODEL, 12
        )

        self.family_head = nn.Linear(
            D_MODEL, 7
        )

    def forward(self, batch):

        # Melody encoding

        melody = (
            self.pitch_embedding(batch["melody_pitch"])
            +
            self.melody_time_projection(batch["melody_time"])
        )

        melody = add_positional_encoding(melody)

        melody = self.dropout(
            self.melody_norm(melody)
        )

        melody = melody.masked_fill(
            batch["melody_padding_mask"].unsqueeze(-1),
            0.0
        )

        memory = self.encoder(
            melody,
            src_key_padding_mask=batch[
                "melody_padding_mask"
            ]
        )

        # Beat queries — no previous chord inputs

        beats = (
            self.beat_time_projection(batch["beat_time"])
            +
            self.bar_embedding(batch["bar_index"])
            +
            self.beat_position_embedding(
                batch["beat_position"]
            )
            +
            self.meter_embedding(batch["meter"])
        )

        beats = add_positional_encoding(beats)

        beats = self.dropout(
            self.beat_norm(beats)
        )

        beats = beats.masked_fill(
            batch["beat_padding_mask"].unsqueeze(-1),
            0.0
        )

        temporal_bias = build_temporal_bias(batch)

        decoded = self.decoder(
            tgt=beats,
            memory=memory,
            memory_mask=temporal_bias,
            tgt_key_padding_mask=batch[
                "beat_padding_mask"
            ]
        )

        return (
            self.root_head(decoded),
            self.family_head(decoded)
        )


# ============================================================
# 6. Class Weights — Computed from V3 Training Data
# ============================================================

root_counts = torch.zeros(12)
family_counts = torch.zeros(7)

for ex in train_examples:

    targets = ex["v3_targets"]

    for root, family, valid in zip(
        targets["root_ids"],
        targets["family_ids"],
        targets["chord_mask"]
    ):

        if valid:
            root_counts[root] += 1
            family_counts[family] += 1


def calculate_weights(counts):

    present = counts > 0

    weights = torch.zeros_like(counts)

    mean_count = counts[present].mean()

    weights[present] = (
        mean_count / counts[present]
    ).sqrt()

    average = (
        (weights * counts).sum()
        / counts.sum()
    )

    weights[present] /= average

    weights[present] = weights[present].clamp(
        min=0.5,
        max=4.0
    )

    return weights.to(DEVICE)


root_weights = calculate_weights(root_counts)
family_weights = calculate_weights(family_counts)


# ============================================================
# 7. Training and Validation Functions
# ============================================================

def move_batch(batch):

    return {
        key: (
            value.to(DEVICE)
            if isinstance(value, torch.Tensor)
            else value
        )
        for key, value in batch.items()
    }


def calculate_loss(batch, root_logits, family_logits,
                   weighted=True):

    rw = root_weights if weighted else None
    fw = family_weights if weighted else None

    root_loss = F.cross_entropy(
        root_logits.reshape(-1, 12),
        batch["target_root"].reshape(-1),
        weight=rw,
        ignore_index=-100
    )

    family_loss = F.cross_entropy(
        family_logits.reshape(-1, 7),
        batch["target_family"].reshape(-1),
        weight=fw,
        ignore_index=-100
    )

    return root_loss + family_loss


def macro_f1(confusion):

    matrix = confusion.float()

    tp = matrix.diag()

    actual = matrix.sum(dim=1)
    predicted = matrix.sum(dim=0)

    f1 = (
        2 * tp
        / (actual + predicted).clamp(min=1)
    )

    return f1.mean().item()


def train_epoch(model, optimizer):

    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in train_loader:

        batch = move_batch(batch)

        optimizer.zero_grad(set_to_none=True)

        root_logits, family_logits = model(batch)

        loss = calculate_loss(
            batch,
            root_logits,
            family_logits,
            weighted=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / total_batches


@torch.no_grad()
def validate_epoch(model):

    model.eval()

    root_cm = torch.zeros(
        12, 12, dtype=torch.long
    )

    family_cm = torch.zeros(
        7, 7, dtype=torch.long
    )

    total_loss = 0.0
    total_valid = 0
    joint_correct = 0

    change_tp = 0
    change_fp = 0
    change_fn = 0

    for batch in val_loader:

        batch = move_batch(batch)

        root_logits, family_logits = model(batch)

        valid = batch["target_root"] != -100

        n = int(valid.sum().item())

        # Unweighted CE for checkpoint selection

        loss = calculate_loss(
            batch,
            root_logits,
            family_logits,
            weighted=False
        )

        total_loss += loss.item() * n
        total_valid += n

        pred_root = root_logits.argmax(dim=-1)
        pred_family = family_logits.argmax(dim=-1)

        true_root = batch["target_root"]
        true_family = batch["target_family"]

        # Root confusion matrix

        root_cm += torch.bincount(
            (
                true_root[valid] * 12
                + pred_root[valid]
            ).cpu(),
            minlength=144
        ).reshape(12, 12)

        # Family confusion matrix

        family_cm += torch.bincount(
            (
                true_family[valid] * 7
                + pred_family[valid]
            ).cpu(),
            minlength=49
        ).reshape(7, 7)

        joint_correct += int(
            (
                (pred_root == true_root)
                &
                (pred_family == true_family)
                &
                valid
            ).sum().item()
        )

        # Chord-change detection:
        # same adjacent-valid-beat definition
        # as the original chord-model evaluation.

        pair_valid = (
            valid[:, 1:] & valid[:, :-1]
        )

        actual_change = (
            (true_root[:, 1:] != true_root[:, :-1])
            |
            (true_family[:, 1:] != true_family[:, :-1])
        )

        predicted_change = (
            (pred_root[:, 1:] != pred_root[:, :-1])
            |
            (pred_family[:, 1:] != pred_family[:, :-1])
        )

        change_tp += int(
            (
                predicted_change
                & actual_change
                & pair_valid
            ).sum().item()
        )

        change_fp += int(
            (
                predicted_change
                & ~actual_change
                & pair_valid
            ).sum().item()
        )

        change_fn += int(
            (
                ~predicted_change
                & actual_change
                & pair_valid
            ).sum().item()
        )

    return {
        "val_loss": total_loss / total_valid,

        "root_acc":
            root_cm.diag().sum().item() / total_valid,

        "family_acc":
            family_cm.diag().sum().item() / total_valid,

        "joint_acc":
            joint_correct / total_valid,

        "root_f1": macro_f1(root_cm),

        "family_f1": macro_f1(family_cm),

        "change_f1": (
            2 * change_tp
            / max(
                2 * change_tp + change_fp + change_fn,
                1
            )
        )
    }


# ============================================================
# 8. Initialize Fresh Model
# ============================================================

model1 = HarmonyTemporalTransformer().to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model1.parameters()
    if p.requires_grad
)

assert parameter_count == 950931

optimizer = torch.optim.AdamW(
    model1.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 9. Train
# ============================================================

history = []

best_loss = float("inf")
best_epoch = None
epochs_without_improvement = 0

print("\nMODEL 1 — IMPROVED TEMPORAL TRANSFORMER")
print("--------------------------------------")
print("Device:", DEVICE)
print("Training windows:", len(train_examples))
print("Validation windows:", len(val_examples))
print("Parameters:", f"{parameter_count:,}")
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping:", PATIENCE)
print("Temporal sigma:", TEMPORAL_SIGMA)
print("\nTraining started...\n")


for epoch in range(1, MAX_EPOCHS + 1):

    start_time = time.perf_counter()

    train_loss = train_epoch(
        model1, optimizer
    )

    metrics = validate_epoch(model1)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **metrics
    }

    history.append(row)

    improved = (
        metrics["val_loss"] < best_loss - 1e-4
    )

    if improved:

        best_loss = metrics["val_loss"]
        best_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model1.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_metrics": metrics,
                "experiment": EXPERIMENT_NAME,
                "dataset": DATASET_PATH.name,
                "temporal_sigma": TEMPORAL_SIGMA,
                "temporal_strength": TEMPORAL_STRENGTH,
                "parameter_count": parameter_count,
                "seed": SEED
            },
            CHECKPOINT_PATH
        )

    else:

        epochs_without_improvement += 1

    elapsed = time.perf_counter() - start_time

    print(
        f"Epoch {epoch:02d} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {metrics['val_loss']:.4f} | "
        f"Root: {metrics['root_acc']:.2%} | "
        f"Family: {metrics['family_acc']:.2%} | "
        f"Joint: {metrics['joint_acc']:.2%} | "
        f"Root F1: {metrics['root_f1']:.4f} | "
        f"Change F1: {metrics['change_f1']:.4f} | "
        f"{elapsed:.1f}s"
        + (" | BEST" if improved else "")
    )

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break


# ============================================================
# 10. Load Best Checkpoint and Compare
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model1.load_state_dict(
    checkpoint["model_state_dict"]
)

best_metrics = validate_epoch(model1)

history_df = pd.DataFrame(history)

print("\n" + "=" * 55)
print("FINAL MODEL 1 RESULTS")
print("=" * 55)

print("Best epoch:", best_epoch)
print("Checkpoint:", CHECKPOINT_PATH.name)

print(
    f"Root accuracy:   {best_metrics['root_acc']:.2%}"
)

print(
    f"Family accuracy: {best_metrics['family_acc']:.2%}"
)

print(
    f"Joint accuracy:  {best_metrics['joint_acc']:.2%}"
)

print(
    f"Root macro-F1:   {best_metrics['root_f1']:.4f}"
)

print(
    f"Family macro-F1: {best_metrics['family_f1']:.4f}"
)

print(
    f"Change F1:       {best_metrics['change_f1']:.4f}"
)


# Compare against original temporal Transformer.

BASELINE_JOINT = 0.3096

difference = (
    best_metrics["joint_acc"] - BASELINE_JOINT
) * 100

print("\nCOMPARISON WITH PREVIOUS TEMPORAL MODEL")
print("--------------------------------------")

print("Previous joint accuracy: 30.96%")

print(
    "New joint accuracy:",
    f"{best_metrics['joint_acc']:.2%}"
)

print(
    "Difference:",
    f"{difference:+.2f} percentage points"
)

print("\nModel checkpoint saved successfully.")


MODEL 1 — IMPROVED TEMPORAL TRANSFORMER
--------------------------------------
Device: cpu
Training windows: 4523
Validation windows: 1405
Parameters: 950,931
Maximum epochs: 20
Early stopping: 5
Temporal sigma: 2.0

Training started...

Epoch 01 | Train: 3.1562 | Val: 2.7231 | Root: 34.34% | Family: 54.72% | Joint: 25.11% | Root F1: 0.1589 | Change F1: 0.1451 | 16.0s | BEST
Epoch 02 | Train: 2.9881 | Val: 2.5803 | Root: 36.98% | Family: 58.25% | Joint: 28.29% | Root F1: 0.1813 | Change F1: 0.1405 | 15.2s | BEST
Epoch 03 | Train: 2.9316 | Val: 2.6407 | Root: 36.87% | Family: 57.17% | Joint: 25.38% | Root F1: 0.1892 | Change F1: 0.1493 | 15.1s
Epoch 04 | Train: 2.8900 | Val: 2.6138 | Root: 36.19% | Family: 56.32% | Joint: 27.45% | Root F1: 0.1926 | Change F1: 0.1542 | 14.9s
Epoch 05 | Train: 2.8560 | Val: 2.6236 | Root: 34.76% | Family: 58.58% | Joint: 26.36% | Root F1: 0.1788 | Change F1: 0.1510 | 15.2s
Epoch 06 | Train: 2.8089 | Val: 2.7473 | Root: 32.55% | Family: 55.26% | Joint: 23

### Train Boundary-Aware Temporal Transformer

In [3]:
# ============================================================
# MODEL 2 — BOUNDARY-AWARE TEMPORAL TRANSFORMER
# ============================================================

import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

EXPERIMENT_2 = "phase3_v3_boundary_aware_temporal"

CHANGE_LOSS_WEIGHT = 0.5
CHANGE_THRESHOLD = 0.5

MODEL2_CHECKPOINT = (
    CHECKPOINT_DIR
    / f"{EXPERIMENT_2}_best.pt"
)

# Calculate change-class weighting from training data.

change_counts = torch.zeros(2, dtype=torch.long)

for ex in train_examples:

    labels = ex["v3_targets"]["change_labels"]

    for label in labels:
        if label in (0, 1):
            change_counts[label] += 1

assert change_counts.tolist() == [198800, 54422]

change_pos_weight = torch.sqrt(
    change_counts[0].float()
    / change_counts[1].float()
).to(DEVICE)

print("CHANGE CLASS BALANCING")
print("--------------------------------")
print("No change:", change_counts[0].item())
print("Change:", change_counts[1].item())
print("Positive weight:", round(change_pos_weight.item(), 4))
print("Change loss weight:", CHANGE_LOSS_WEIGHT)


# ============================================================
# 2. Boundary-Aware Architecture
# ============================================================

class BoundaryAwareTemporalTransformer(
    HarmonyTemporalTransformer
):

    def __init__(self):

        super().__init__()

        # Additional prediction head.
        # The original root/family architecture
        # remains unchanged.

        self.change_head = nn.Linear(
            D_MODEL, 1
        )

    def forward(self, batch):

        # ------------------------------------
        # A. Melody encoding
        # ------------------------------------

        melody = (
            self.pitch_embedding(
                batch["melody_pitch"]
            )
            +
            self.melody_time_projection(
                batch["melody_time"]
            )
        )

        melody = add_positional_encoding(melody)

        melody = self.dropout(
            self.melody_norm(melody)
        )

        melody = melody.masked_fill(
            batch["melody_padding_mask"].unsqueeze(-1),
            0.0
        )

        memory = self.encoder(
            melody,
            src_key_padding_mask=batch[
                "melody_padding_mask"
            ]
        )

        # ------------------------------------
        # B. Beat-level decoder queries
        # ------------------------------------

        beats = (
            self.beat_time_projection(
                batch["beat_time"]
            )
            +
            self.bar_embedding(
                batch["bar_index"]
            )
            +
            self.beat_position_embedding(
                batch["beat_position"]
            )
            +
            self.meter_embedding(
                batch["meter"]
            )
        )

        beats = add_positional_encoding(beats)

        beats = self.dropout(
            self.beat_norm(beats)
        )

        beats = beats.masked_fill(
            batch["beat_padding_mask"].unsqueeze(-1),
            0.0
        )

        # ------------------------------------
        # C. Gaussian temporal attention
        # ------------------------------------

        temporal_bias = build_temporal_bias(batch)

        decoded = self.decoder(
            tgt=beats,
            memory=memory,
            memory_mask=temporal_bias,
            tgt_key_padding_mask=batch[
                "beat_padding_mask"
            ]
        )

        # ------------------------------------
        # D. Three prediction heads
        # ------------------------------------

        root_logits = self.root_head(decoded)

        family_logits = self.family_head(decoded)

        change_logits = (
            self.change_head(decoded).squeeze(-1)
        )

        return (
            root_logits,
            family_logits,
            change_logits
        )


# ============================================================
# 3. Multi-Task Loss
# ============================================================

def model2_loss(
    batch,
    root_logits,
    family_logits,
    change_logits,
    weighted=True
):

    # Root and family classification.
    # Reuses Model 1's class weights.

    chord_loss = calculate_loss(
        batch,
        root_logits,
        family_logits,
        weighted=weighted
    )

    # Independent event-based change labels.

    change_targets = batch["change_labels"]

    change_valid = (
        change_targets != IGNORE_INDEX
    )

    if change_valid.any():

        logits = change_logits[change_valid]

        targets = (
            change_targets[change_valid].float()
        )

        change_loss = (
            F.binary_cross_entropy_with_logits(
                logits,
                targets,
                pos_weight=(
                    change_pos_weight
                    if weighted
                    else None
                )
            )
        )

    else:

        change_loss = (
            change_logits.sum() * 0.0
        )

    total_loss = (
        chord_loss
        +
        CHANGE_LOSS_WEIGHT * change_loss
    )

    return total_loss, chord_loss, change_loss


# ============================================================
# 4. Training Function
# ============================================================

def train_model2_epoch(model, optimizer):

    model.train()

    total_loss = 0.0
    total_chord = 0.0
    total_change = 0.0

    for batch in train_loader:

        batch = move_batch(batch)

        optimizer.zero_grad(
            set_to_none=True
        )

        root_logits, family_logits, change_logits = (
            model(batch)
        )

        loss, chord_loss, change_loss = (
            model2_loss(
                batch,
                root_logits,
                family_logits,
                change_logits,
                weighted=True
            )
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        optimizer.step()

        total_loss += loss.item()
        total_chord += chord_loss.item()
        total_change += change_loss.item()

    n = len(train_loader)

    return {
        "train_loss": total_loss / n,
        "train_chord_loss": total_chord / n,
        "train_change_loss": total_change / n
    }


# ============================================================
# 5. Validation Function
# ============================================================

@torch.no_grad()
def validate_model2(model):

    model.eval()

    root_cm = torch.zeros(
        12, 12, dtype=torch.long
    )

    family_cm = torch.zeros(
        7, 7, dtype=torch.long
    )

    total_chord_loss = 0.0
    total_change_loss = 0.0

    total_chords = 0
    total_changes = 0

    joint_correct = 0

    # Explicit boundary-head evaluation.

    tp = 0
    fp = 0
    fn = 0

    predicted_positives = 0
    actual_positives = 0

    # Original adjacent-valid chord-change metric.
    # Kept separate from event-based boundary F1.

    implicit_tp = 0
    implicit_fp = 0
    implicit_fn = 0

    for batch in val_loader:

        batch = move_batch(batch)

        root_logits, family_logits, change_logits = (
            model(batch)
        )

        # ------------------------------------
        # A. Chord classification
        # ------------------------------------

        root_targets = batch["target_root"]
        family_targets = batch["target_family"]

        valid = (
            (root_targets != IGNORE_INDEX)
            &
            (family_targets != IGNORE_INDEX)
        )

        n_chords = int(valid.sum().item())

        chord_loss = calculate_loss(
            batch,
            root_logits,
            family_logits,
            weighted=False
        )

        total_chord_loss += (
            chord_loss.item() * n_chords
        )

        total_chords += n_chords

        pred_root = (
            root_logits.argmax(dim=-1)
        )

        pred_family = (
            family_logits.argmax(dim=-1)
        )

        root_cm += torch.bincount(
            (
                root_targets[valid] * 12
                + pred_root[valid]
            ).cpu(),
            minlength=144
        ).reshape(12, 12)

        family_cm += torch.bincount(
            (
                family_targets[valid] * 7
                + pred_family[valid]
            ).cpu(),
            minlength=49
        ).reshape(7, 7)

        joint_correct += int(
            (
                (pred_root == root_targets)
                &
                (pred_family == family_targets)
                &
                valid
            ).sum().item()
        )

        # ------------------------------------
        # B. Explicit event-based boundaries
        # ------------------------------------

        change_targets = batch[
            "change_labels"
        ]

        change_valid = (
            change_targets != IGNORE_INDEX
        )

        n_changes = int(
            change_valid.sum().item()
        )

        if n_changes:

            true_changes = (
                change_targets[change_valid]
            )

            change_scores = (
                change_logits[change_valid]
            )

            change_loss = (
                F.binary_cross_entropy_with_logits(
                    change_scores,
                    true_changes.float()
                )
            )

            total_change_loss += (
                change_loss.item() * n_changes
            )

            total_changes += n_changes

            predicted_changes = (
                torch.sigmoid(change_scores)
                >= CHANGE_THRESHOLD
            )

            actual_changes = (
                true_changes == 1
            )

            tp += int(
                (
                    predicted_changes
                    & actual_changes
                ).sum().item()
            )

            fp += int(
                (
                    predicted_changes
                    & ~actual_changes
                ).sum().item()
            )

            fn += int(
                (
                    ~predicted_changes
                    & actual_changes
                ).sum().item()
            )

            predicted_positives += int(
                predicted_changes.sum().item()
            )

            actual_positives += int(
                actual_changes.sum().item()
            )

        # ------------------------------------
        # C. Old implicit change metric
        # ------------------------------------

        pair_valid = (
            valid[:, 1:]
            &
            valid[:, :-1]
        )

        actual_pair_change = (
            (root_targets[:, 1:] != root_targets[:, :-1])
            |
            (family_targets[:, 1:] != family_targets[:, :-1])
        )

        predicted_pair_change = (
            (pred_root[:, 1:] != pred_root[:, :-1])
            |
            (pred_family[:, 1:] != pred_family[:, :-1])
        )

        implicit_tp += int(
            (
                pair_valid
                & actual_pair_change
                & predicted_pair_change
            ).sum().item()
        )

        implicit_fp += int(
            (
                pair_valid
                & ~actual_pair_change
                & predicted_pair_change
            ).sum().item()
        )

        implicit_fn += int(
            (
                pair_valid
                & actual_pair_change
                & ~predicted_pair_change
            ).sum().item()
        )

    # ----------------------------------------
    # D. Final metrics
    # ----------------------------------------

    root_accuracy = (
        root_cm.diag().sum().item()
        / total_chords
    )

    family_accuracy = (
        family_cm.diag().sum().item()
        / total_chords
    )

    joint_accuracy = (
        joint_correct / total_chords
    )

    precision = (
        tp / max(tp + fp, 1)
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    change_f1 = (
        2 * tp
        / max(2 * tp + fp + fn, 1)
    )

    implicit_f1 = (
        2 * implicit_tp
        / max(
            2 * implicit_tp
            + implicit_fp
            + implicit_fn,
            1
        )
    )

    val_chord_ce = (
        total_chord_loss / total_chords
    )

    val_change_bce = (
        total_change_loss / total_changes
    )

    # Joint validation objective.
    # Both terms are unweighted here.
    # Lower is better.

    selection_loss = (
        val_chord_ce
        +
        CHANGE_LOSS_WEIGHT * val_change_bce
    )

    return {

        "selection_loss": selection_loss,

        "val_chord_ce": val_chord_ce,

        "val_change_bce": val_change_bce,

        "root_acc": root_accuracy,

        "family_acc": family_accuracy,

        "joint_acc": joint_accuracy,

        "root_f1": macro_f1(root_cm),

        "family_f1": macro_f1(family_cm),

        "event_change_precision": precision,

        "event_change_recall": recall,

        "event_change_f1": change_f1,

        "implicit_change_f1": implicit_f1,

        "predicted_change_rate": (
            predicted_positives / total_changes
        ),

        "actual_change_rate": (
            actual_positives / total_changes
        )
    }


# ============================================================
# 6. Initialize Fresh Model
# ============================================================

torch.manual_seed(SEED)

model2 = (
    BoundaryAwareTemporalTransformer()
    .to(DEVICE)
)

model2_parameters = sum(
    p.numel()
    for p in model2.parameters()
    if p.requires_grad
)

# Original model: 950,931
# Added binary head: 128 weights + 1 bias

assert model2_parameters == 951060

optimizer2 = torch.optim.AdamW(
    model2.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 7. Train Model 2
# ============================================================

model2_history = []

best_selection_loss = float("inf")
best_epoch2 = None

epochs_without_improvement = 0

print("\nMODEL 2 — BOUNDARY-AWARE TEMPORAL TRANSFORMER")
print("------------------------------------------------")
print("Device:", DEVICE)
print("Training windows:", len(train_examples))
print("Validation windows:", len(val_examples))
print("Parameters:", f"{model2_parameters:,}")
print("Maximum epochs:", MAX_EPOCHS)
print("Change loss weight:", CHANGE_LOSS_WEIGHT)
print("Change positive weight:", round(change_pos_weight.item(), 4))
print("\nTraining started...\n")


for epoch in range(1, MAX_EPOCHS + 1):

    start_time = time.perf_counter()

    train_metrics = train_model2_epoch(
        model2,
        optimizer2
    )

    val_metrics = validate_model2(
        model2
    )

    model2_history.append({
        "epoch": epoch,
        **train_metrics,
        **val_metrics
    })

    improved = (
        val_metrics["selection_loss"]
        <
        best_selection_loss - 1e-4
    )

    if improved:

        best_selection_loss = (
            val_metrics["selection_loss"]
        )

        best_epoch2 = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model2.state_dict(),
                "optimizer_state_dict": optimizer2.state_dict(),
                "validation_metrics": val_metrics,
                "experiment": EXPERIMENT_2,
                "dataset": DATASET_PATH.name,
                "change_loss_weight": CHANGE_LOSS_WEIGHT,
                "change_pos_weight": change_pos_weight.item(),
                "parameter_count": model2_parameters,
                "seed": SEED
            },
            MODEL2_CHECKPOINT
        )

    else:

        epochs_without_improvement += 1

    elapsed = (
        time.perf_counter() - start_time
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train: {train_metrics['train_loss']:.4f} | "
        f"Val: {val_metrics['selection_loss']:.4f} | "
        f"Joint: {val_metrics['joint_acc']:.2%} | "
        f"Root F1: {val_metrics['root_f1']:.4f} | "
        f"Event Change F1: {val_metrics['event_change_f1']:.4f} | "
        f"Precision: {val_metrics['event_change_precision']:.4f} | "
        f"Recall: {val_metrics['event_change_recall']:.4f} | "
        f"Implicit Change F1: {val_metrics['implicit_change_f1']:.4f} | "
        f"{elapsed:.1f}s"
        + (" | BEST" if improved else "")
    )

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )
        break


# ============================================================
# 8. Restore Best Checkpoint
# ============================================================

checkpoint2 = torch.load(
    MODEL2_CHECKPOINT,
    map_location=DEVICE,
    weights_only=True
)

model2.load_state_dict(
    checkpoint2["model_state_dict"]
)

best_metrics2 = validate_model2(
    model2
)

model2_history_df = pd.DataFrame(
    model2_history
)


# ============================================================
# 9. Final Results
# ============================================================

print("\n" + "=" * 60)
print("FINAL MODEL 2 RESULTS")
print("=" * 60)

print("Best epoch:", best_epoch2)

print(
    f"Root accuracy:             "
    f"{best_metrics2['root_acc']:.2%}"
)

print(
    f"Family accuracy:           "
    f"{best_metrics2['family_acc']:.2%}"
)

print(
    f"Joint chord accuracy:      "
    f"{best_metrics2['joint_acc']:.2%}"
)

print(
    f"Root macro-F1:             "
    f"{best_metrics2['root_f1']:.4f}"
)

print(
    f"Family macro-F1:           "
    f"{best_metrics2['family_f1']:.4f}"
)

print(
    f"Event-based change F1:     "
    f"{best_metrics2['event_change_f1']:.4f}"
)

print(
    f"Event change precision:    "
    f"{best_metrics2['event_change_precision']:.4f}"
)

print(
    f"Event change recall:       "
    f"{best_metrics2['event_change_recall']:.4f}"
)

print(
    f"Implicit chord-change F1:  "
    f"{best_metrics2['implicit_change_f1']:.4f}"
)

print(
    f"Predicted event change rate: "
    f"{best_metrics2['predicted_change_rate']:.2%}"
)

print(
    f"Actual event change rate:    "
    f"{best_metrics2['actual_change_rate']:.2%}"
)


# ============================================================
# 10. Compare Chord Accuracy with Model 1
# ============================================================

model1_joint = best_metrics["joint_acc"]

model2_joint = best_metrics2["joint_acc"]

print("\nMODEL 1 VS MODEL 2")
print("--------------------------------")

comparison = pd.DataFrame([
    {
        "Model": "Model 1 — Temporal",
        "Joint accuracy": model1_joint,
        "Root accuracy": best_metrics["root_acc"],
        "Root macro-F1": best_metrics["root_f1"]
    },
    {
        "Model": "Model 2 — Boundary-Aware",
        "Joint accuracy": model2_joint,
        "Root accuracy": best_metrics2["root_acc"],
        "Root macro-F1": best_metrics2["root_f1"]
    }
])

display(comparison.round(4))

print(
    "\nJoint accuracy difference:",
    f"{(model2_joint - model1_joint) * 100:+.2f}",
    "percentage points"
)

print("\nModel 2 checkpoint saved:")
print(MODEL2_CHECKPOINT)

CHANGE CLASS BALANCING
--------------------------------
No change: 198800
Change: 54422
Positive weight: 1.9113
Change loss weight: 0.5

MODEL 2 — BOUNDARY-AWARE TEMPORAL TRANSFORMER
------------------------------------------------
Device: cpu
Training windows: 4523
Validation windows: 1405
Parameters: 951,060
Maximum epochs: 20
Change loss weight: 0.5
Change positive weight: 1.9113

Training started...

Epoch 01 | Train: 3.5426 | Val: 3.0540 | Joint: 23.48% | Root F1: 0.1789 | Event Change F1: 0.0057 | Precision: 0.2485 | Recall: 0.0029 | Implicit Change F1: 0.1614 | 15.5s | BEST
Epoch 02 | Train: 3.3701 | Val: 2.9465 | Joint: 26.40% | Root F1: 0.1869 | Event Change F1: 0.1218 | Precision: 0.3130 | Recall: 0.0756 | Implicit Change F1: 0.1519 | 15.4s | BEST
Epoch 03 | Train: 3.3121 | Val: 2.8590 | Joint: 27.12% | Root F1: 0.1931 | Event Change F1: 0.1075 | Precision: 0.3217 | Recall: 0.0645 | Implicit Change F1: 0.1580 | 15.1s | BEST
Epoch 04 | Train: 3.2648 | Val: 2.9114 | Joint: 26.6

,Model,Joint accuracy,Root accuracy,Root macro-F1
0,Model 1 — Temporal,0.2829,0.3698,0.1813
1,Model 2 — Boundary-Aware,0.2712,0.3599,0.1931



Joint accuracy difference: -1.18 percentage points

Model 2 checkpoint saved:
/Users/praise/Developer/vocal-harmonization/experiments/pasan/checkpoints/phase3_v3_boundary_aware_temporal_best.pt


In [4]:
# ============================================================
# MODEL 2B — DEEP BOUNDARY-AWARE TRANSFORMER
# 3 encoder + 3 decoder layers
# 60 epochs, NO early stopping
# Stronger change supervision + threshold calibration
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# 1. Experimental configuration
# ============================================================

DEEP_EPOCHS = 60

DEEP_ENCODER_LAYERS = 3
DEEP_DECODER_LAYERS = 3

DEEP_CHANGE_LOSS_WEIGHT = 1.0

DEEP_CHANGE_POS_WEIGHT = (
    change_counts[0].float()
    / change_counts[1].float()
).to(DEVICE)

DEEP_LEARNING_RATE = 3e-4
DEEP_MIN_LEARNING_RATE = 1e-5

DEEP_EXPERIMENT = "phase3_v3_deep_boundary_aware"

DEEP_COMBINED_PATH = (
    CHECKPOINT_DIR
    / f"{DEEP_EXPERIMENT}_best_combined.pt"
)

DEEP_CHORD_PATH = (
    CHECKPOINT_DIR
    / f"{DEEP_EXPERIMENT}_best_chord.pt"
)


# ============================================================
# 2. Threshold calibration
# ============================================================

@torch.no_grad()
def evaluate_change_thresholds(model, name):

    model.eval()

    all_probabilities = []
    all_labels = []

    for batch in val_loader:

        batch = move_batch(batch)

        _, _, change_logits = model(batch)

        labels = batch["change_labels"]

        valid = labels != IGNORE_INDEX

        probabilities = torch.sigmoid(
            change_logits[valid]
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

        all_labels.append(
            labels[valid].cpu().numpy()
        )

    probabilities = np.concatenate(
        all_probabilities
    )

    labels = np.concatenate(all_labels)

    actual_positive = labels == 1

    rows = []

    # Includes 0.0 (always positive)
    # and 0.5 (our original threshold).

    thresholds = np.linspace(
        0.0, 0.95, 39
    )

    for threshold in thresholds:

        predicted_positive = (
            probabilities >= threshold
        )

        tp = np.sum(
            predicted_positive & actual_positive
        )

        fp = np.sum(
            predicted_positive & ~actual_positive
        )

        fn = np.sum(
            ~predicted_positive & actual_positive
        )

        precision = (
            tp / max(tp + fp, 1)
        )

        recall = (
            tp / max(tp + fn, 1)
        )

        f1 = (
            2 * tp
            / max(2 * tp + fp + fn, 1)
        )

        rows.append({
            "threshold": float(threshold),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "predicted_rate":
                predicted_positive.mean()
        })

    results = pd.DataFrame(rows)

    best = results.loc[
        results["f1"].idxmax()
    ]

    at_half = results.iloc[
        (results["threshold"] - 0.5)
        .abs()
        .argmin()
    ]

    always_positive_f1 = results.iloc[0]["f1"]

    print(f"\n{name}")
    print("-" * 45)

    print(
        "F1 at threshold 0.5:",
        f"{at_half['f1']:.4f}"
    )

    print(
        "Best validation threshold:",
        f"{best['threshold']:.3f}"
    )

    print(
        "Best validation F1:",
        f"{best['f1']:.4f}"
    )

    print(
        "Precision:",
        f"{best['precision']:.4f}"
    )

    print(
        "Recall:",
        f"{best['recall']:.4f}"
    )

    print(
        "Predicted change rate:",
        f"{best['predicted_rate']:.2%}"
    )

    print(
        "Always-positive baseline F1:",
        f"{always_positive_f1:.4f}"
    )

    return results, best.to_dict()


# Check whether the original Model 2 was
# affected by its 0.5 decision threshold.

print("\nORIGINAL MODEL 2 — THRESHOLD ANALYSIS")

model2_threshold_df, model2_best_threshold = (
    evaluate_change_thresholds(
        model2,
        "Original Model 2"
    )
)


# ============================================================
# 3. Deeper Transformer architecture
# ============================================================

class DeepBoundaryAwareTransformer(
    BoundaryAwareTemporalTransformer
):

    def __init__(self):

        super().__init__()

        # Replace original 2-layer encoder
        # with a 3-layer encoder.

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=DEEP_ENCODER_LAYERS,
            norm=nn.LayerNorm(D_MODEL),
            enable_nested_tensor=False
        )

        # Replace original 2-layer decoder
        # with a 3-layer decoder.

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=DEEP_DECODER_LAYERS,
            norm=nn.LayerNorm(D_MODEL)
        )

        # Root, family and change heads
        # are inherited unchanged.


# ============================================================
# 4. Initialize Model 2B
# ============================================================

torch.manual_seed(SEED)

model2b = DeepBoundaryAwareTransformer().to(
    DEVICE
)

model2b_parameters = sum(
    p.numel()
    for p in model2b.parameters()
    if p.requires_grad
)

optimizer2b = torch.optim.AdamW(
    model2b.parameters(),
    lr=DEEP_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Gradually reduce LR over 60 epochs.
# This allows longer, more stable training.

scheduler2b = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer2b,
        T_max=DEEP_EPOCHS,
        eta_min=DEEP_MIN_LEARNING_RATE
    )
)


# ============================================================
# 5. Training configuration
# ============================================================

print("\n" + "=" * 60)
print("MODEL 2B — DEEP BOUNDARY-AWARE TRANSFORMER")
print("=" * 60)

print("Device:", DEVICE)
print("Training windows:", len(train_examples))
print("Validation windows:", len(val_examples))

print("Encoder layers:", DEEP_ENCODER_LAYERS)
print("Decoder layers:", DEEP_DECODER_LAYERS)

print("Attention heads:", NHEAD)
print("Embedding dimension:", D_MODEL)

print("Trainable parameters:", f"{model2b_parameters:,}")

print("Epochs:", DEEP_EPOCHS)
print("Early stopping: DISABLED")

print(
    "Change-positive weight:",
    round(DEEP_CHANGE_POS_WEIGHT.item(), 4)
)

print(
    "Change-loss weight:",
    DEEP_CHANGE_LOSS_WEIGHT
)

print("Initial LR:", DEEP_LEARNING_RATE)
print("Minimum LR:", DEEP_MIN_LEARNING_RATE)


# ============================================================
# 6. Train for ALL 60 epochs
# ============================================================

deep_history = []

best_combined_loss = float("inf")
best_chord_loss = float("inf")

best_combined_epoch = None
best_chord_epoch = None

# The existing Model 2 loss functions use
# these global variables.
# Temporarily override them for this experiment,
# then restore their original values.

original_change_loss_weight = CHANGE_LOSS_WEIGHT
original_change_pos_weight = change_pos_weight

try:

    CHANGE_LOSS_WEIGHT = (
        DEEP_CHANGE_LOSS_WEIGHT
    )

    change_pos_weight = (
        DEEP_CHANGE_POS_WEIGHT
    )

    print("\nTraining started...\n")

    for epoch in range(1, DEEP_EPOCHS + 1):

        start_time = time.perf_counter()

        # ------------------------------------
        # Train
        # ------------------------------------

        train_metrics = train_model2_epoch(
            model2b,
            optimizer2b
        )

        # ------------------------------------
        # Validate
        # ------------------------------------

        val_metrics = validate_model2(
            model2b
        )

        current_lr = (
            optimizer2b.param_groups[0]["lr"]
        )

        scheduler2b.step()

        # ------------------------------------
        # Record history
        # ------------------------------------

        deep_history.append({
            "epoch": epoch,
            "lr": current_lr,
            **train_metrics,
            **val_metrics
        })

        # ------------------------------------
        # Save best combined-loss checkpoint
        # ------------------------------------

        combined_improved = (
            val_metrics["selection_loss"]
            < best_combined_loss
        )

        if combined_improved:

            best_combined_loss = (
                val_metrics["selection_loss"]
            )

            best_combined_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict":
                        model2b.state_dict(),
                    "optimizer_state_dict":
                        optimizer2b.state_dict(),
                    "validation_metrics":
                        val_metrics,
                    "parameters":
                        model2b_parameters,
                    "encoder_layers":
                        DEEP_ENCODER_LAYERS,
                    "decoder_layers":
                        DEEP_DECODER_LAYERS,
                    "change_loss_weight":
                        DEEP_CHANGE_LOSS_WEIGHT,
                    "change_pos_weight":
                        DEEP_CHANGE_POS_WEIGHT.item(),
                    "dataset":
                        DATASET_PATH.name,
                    "seed": SEED
                },
                DEEP_COMBINED_PATH
            )

        # ------------------------------------
        # Save best chord-loss checkpoint
        # ------------------------------------

        # This allows us to examine whether
        # improving changes hurts chord accuracy.

        chord_improved = (
            val_metrics["val_chord_ce"]
            < best_chord_loss
        )

        if chord_improved:

            best_chord_loss = (
                val_metrics["val_chord_ce"]
            )

            best_chord_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict":
                        model2b.state_dict(),
                    "validation_metrics":
                        val_metrics,
                    "change_loss_weight":
                        DEEP_CHANGE_LOSS_WEIGHT,
                    "change_pos_weight":
                        DEEP_CHANGE_POS_WEIGHT.item(),
                    "dataset":
                        DATASET_PATH.name,
                    "seed": SEED
                },
                DEEP_CHORD_PATH
            )

        # ------------------------------------
        # Print epoch results
        # ------------------------------------

        elapsed = (
            time.perf_counter()
            - start_time
        )

        print(
            f"Epoch {epoch:02d}/{DEEP_EPOCHS} | "
            f"Train: {train_metrics['train_loss']:.4f} | "
            f"Val: {val_metrics['selection_loss']:.4f} | "
            f"Joint: {val_metrics['joint_acc']:.2%} | "
            f"Root F1: {val_metrics['root_f1']:.4f} | "
            f"Event F1: {val_metrics['event_change_f1']:.4f} | "
            f"Recall: {val_metrics['event_change_recall']:.3f} | "
            f"Pred Rate: {val_metrics['predicted_change_rate']:.2%} | "
            f"LR: {current_lr:.6f} | "
            f"{elapsed:.1f}s"
            + (
                " | BEST COMBINED"
                if combined_improved else ""
            )
            + (
                " | BEST CHORD"
                if chord_improved else ""
            )
        )

    # No early stopping:
    # all 60 epochs have now completed.

    # ========================================================
    # 7. Evaluate both saved checkpoints
    # ========================================================

    deep_results = []

    for name, path in [
        ("Best combined", DEEP_COMBINED_PATH),
        ("Best chord", DEEP_CHORD_PATH)
    ]:

        checkpoint = torch.load(
            path,
            map_location=DEVICE,
            weights_only=True
        )

        model2b.load_state_dict(
            checkpoint["model_state_dict"]
        )

        metrics = validate_model2(
            model2b
        )

        deep_results.append({
            "Checkpoint": name,
            "Epoch": checkpoint["epoch"],
            "Joint accuracy":
                metrics["joint_acc"],
            "Root accuracy":
                metrics["root_acc"],
            "Family accuracy":
                metrics["family_acc"],
            "Root macro-F1":
                metrics["root_f1"],
            "Event F1 at 0.5":
                metrics["event_change_f1"],
            "Event recall at 0.5":
                metrics["event_change_recall"],
            "Predicted change rate":
                metrics["predicted_change_rate"]
        })

    deep_results_df = pd.DataFrame(
        deep_results
    )

    print("\n" + "=" * 60)
    print("MODEL 2B — FINAL CHECKPOINT COMPARISON")
    print("=" * 60)

    display(
        deep_results_df.round(4)
    )

    # ========================================================
    # 8. Threshold calibration on best combined checkpoint
    # ========================================================

    best_checkpoint = torch.load(
        DEEP_COMBINED_PATH,
        map_location=DEVICE,
        weights_only=True
    )

    model2b.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    deep_threshold_df, deep_best_threshold = (
        evaluate_change_thresholds(
            model2b,
            "Model 2B — Best Combined Checkpoint"
        )
    )

finally:

    # Restore Model 2's original loss settings.

    CHANGE_LOSS_WEIGHT = (
        original_change_loss_weight
    )

    change_pos_weight = (
        original_change_pos_weight
    )


# ============================================================
# 9. Training history
# ============================================================

deep_history_df = pd.DataFrame(
    deep_history
)

print("\nTRAINING SUMMARY")
print("--------------------------------")

print("Completed epochs:", len(deep_history_df))

print(
    "Best combined-loss epoch:",
    best_combined_epoch
)

print(
    "Best chord-loss epoch:",
    best_chord_epoch
)

print(
    "Best combined checkpoint:",
    DEEP_COMBINED_PATH.name
)

print(
    "Best chord checkpoint:",
    DEEP_CHORD_PATH.name
)


# ============================================================
# 10. Compare previous models
# ============================================================

print("\nPREVIOUS MODEL BASELINES")
print("--------------------------------")

print("V2 Temporal joint accuracy: 30.96%")
print("V3 Model 1 joint accuracy: 28.29%")
print("V3 Model 2 joint accuracy: 27.12%")

print("\nModel 2B results are shown above.")

print("\nTraining completed.")
print("Early stopping used: NO")
print("Test partition accessed: NO")


ORIGINAL MODEL 2 — THRESHOLD ANALYSIS

Original Model 2
---------------------------------------------
F1 at threshold 0.5: 0.1075
Best validation threshold: 0.350
Best validation F1: 0.3372
Precision: 0.2374
Recall: 0.5819
Predicted change rate: 43.77%
Always-positive baseline F1: 0.3030

MODEL 2B — DEEP BOUNDARY-AWARE TRANSFORMER
Device: cpu
Training windows: 4523
Validation windows: 1405
Encoder layers: 3
Decoder layers: 3
Attention heads: 4
Embedding dimension: 128
Trainable parameters: 1,413,908
Epochs: 60
Early stopping: DISABLED
Change-positive weight: 3.6529
Change-loss weight: 1.0
Initial LR: 0.0003
Minimum LR: 1e-05

Training started...

Epoch 01/60 | Train: 4.2195 | Val: 3.2883 | Joint: 28.03% | Root F1: 0.1783 | Event F1: 0.3358 | Recall: 0.619 | Pred Rate: 48.02% | LR: 0.000300 | 22.9s | BEST COMBINED | BEST CHORD
Epoch 02/60 | Train: 4.0459 | Val: 3.2897 | Joint: 26.61% | Root F1: 0.1890 | Event F1: 0.3303 | Recall: 0.491 | Pred Rate: 35.26% | LR: 0.000300 | 21.7s
Epoch 0

,Checkpoint,Epoch,Joint accuracy,Root accuracy,Family accuracy,Root macro-F1,Event F1 at 0.5,Event recall at 0.5,Predicted change rate
0,Best combined,4,0.2595,0.3451,0.5704,0.1760,0.3362,0.5352,0.3898
1,Best chord,1,0.2803,0.3645,0.5909,0.1783,0.3358,0.6194,0.4802



Model 2B — Best Combined Checkpoint
---------------------------------------------
F1 at threshold 0.5: 0.3362
Best validation threshold: 0.475
Best validation F1: 0.3399
Precision: 0.2353
Recall: 0.6121
Predicted change rate: 46.45%
Always-positive baseline F1: 0.3030

TRAINING SUMMARY
--------------------------------
Completed epochs: 60
Best combined-loss epoch: 4
Best chord-loss epoch: 1
Best combined checkpoint: phase3_v3_deep_boundary_aware_best_combined.pt
Best chord checkpoint: phase3_v3_deep_boundary_aware_best_chord.pt

PREVIOUS MODEL BASELINES
--------------------------------
V2 Temporal joint accuracy: 30.96%
V3 Model 1 joint accuracy: 28.29%
V3 Model 2 joint accuracy: 27.12%

Model 2B results are shown above.

Training completed.
Early stopping used: NO
Test partition accessed: NO


In [9]:
# ============================================================
# CELL 5 — CHORD PROGRESSION QUALITY COMPARISON
# No training, no test access
# ============================================================

import copy
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. Load the best-chord Model 2B checkpoint
# ============================================================

model2b_chord = copy.deepcopy(model2b)

checkpoint = torch.load(
    DEEP_CHORD_PATH,
    map_location=DEVICE,
    weights_only=True
)

model2b_chord.load_state_dict(
    checkpoint["model_state_dict"]
)

# The original model2b object holds the
# best-combined checkpoint from Cell 4.

models = {
    "Model 1": model1,
    "Model 2": model2,
    "Model 2B Combined": model2b,
    "Model 2B Chord": model2b_chord
}

for model in models.values():
    model.eval()


# ============================================================
# 2. Helper functions
# ============================================================

def chord_string(chord):

    if chord is None:
        return "—"

    root, family = chord

    return (
        f"{ROOT_LABELS[root]}:"
        f"{FAMILY_LABELS[family]}"
    )


def longest_run(sequence, valid):

    longest = 0
    current = 0
    previous = None

    for chord, is_valid in zip(sequence, valid):

        if not is_valid:
            current = 0
            previous = None
            continue

        if chord == previous:
            current += 1
        else:
            current = 1

        previous = chord
        longest = max(longest, current)

    return longest


def bar_progression(sequence, valid, beat_grid):

    result = []

    for bar in range(16):

        chords = [
            sequence[i]
            for i, beat in enumerate(beat_grid)
            if (
                int(beat["bar_index"]) == bar
                and valid[i]
                and sequence[i] is not None
            )
        ]

        if chords:
            main_chord = Counter(chords).most_common(1)[0][0]
            result.append(chord_string(main_chord))
        else:
            result.append("—")

    return " | ".join(result)


# ============================================================
# 3. Evaluation storage
# ============================================================

names = ["Reference"] + list(models)

stats = {
    name: {
        "unique": [],
        "constant": 0,
        "longest": [],
        "changes": 0,
        "pairs": 0,
        "tonic": 0,
        "beats": 0,
        "distribution": Counter(),
        "tp": 0,
        "fp": 0,
        "fn": 0
    }
    for name in names
}

rng = random.Random(42)

example_indices = set(
    rng.sample(
        range(len(val_examples)),
        k=3
    )
)

progression_examples = []

window_offset = 0


# ============================================================
# 4. Generate predictions
# ============================================================

with torch.no_grad():

    for batch in val_loader:

        batch = move_batch(batch)

        predictions = {}

        for name, model in models.items():

            outputs = model(batch)

            root_logits = outputs[0]
            family_logits = outputs[1]

            predictions[name] = (
                root_logits.argmax(-1).cpu().numpy(),
                family_logits.argmax(-1).cpu().numpy()
            )

        true_roots = (
            batch["target_root"].cpu().numpy()
        )

        true_families = (
            batch["target_family"].cpu().numpy()
        )

        batch_size = len(true_roots)

        for b in range(batch_size):

            index = window_offset + b
            record = val_examples[index]

            n = len(record["beat_grid"])

            valid = (
                (true_roots[b, :n] != IGNORE_INDEX)
                &
                (true_families[b, :n] != IGNORE_INDEX)
            )

            reference = [
                (
                    int(true_roots[b, i]),
                    int(true_families[b, i])
                )
                if valid[i] else None
                for i in range(n)
            ]

            all_sequences = {
                "Reference": reference
            }

            for name, (roots, families) in predictions.items():

                all_sequences[name] = [
                    (
                        int(roots[b, i]),
                        int(families[b, i])
                    )
                    for i in range(n)
                ]

            # ----------------------------------------
            # Compare only reliably labeled positions
            # ----------------------------------------

            for name, sequence in all_sequences.items():

                s = stats[name]

                observed = [
                    sequence[i]
                    for i in range(n)
                    if valid[i]
                ]

                if not observed:
                    continue

                unique = len(set(observed))

                s["unique"].append(unique)
                s["constant"] += int(unique == 1)

                s["longest"].append(
                    longest_run(sequence, valid)
                )

                s["beats"] += len(observed)

                s["tonic"] += sum(
                    chord == (0, 0)
                    for chord in observed
                )

                s["distribution"].update(observed)

                # Adjacent reliable beat pairs only.

                for i in range(1, n):

                    if not (valid[i] and valid[i - 1]):
                        continue

                    actual_change = (
                        reference[i] != reference[i - 1]
                    )

                    predicted_change = (
                        sequence[i] != sequence[i - 1]
                    )

                    s["pairs"] += 1
                    s["changes"] += int(predicted_change)

                    if name != "Reference":

                        if predicted_change and actual_change:
                            s["tp"] += 1

                        elif predicted_change and not actual_change:
                            s["fp"] += 1

                        elif not predicted_change and actual_change:
                            s["fn"] += 1

            # ----------------------------------------
            # Save three 16-bar examples
            # ----------------------------------------

            if index in example_indices:

                progression_examples.append({
                    "track": record["track_id"],
                    "window": record["window_index"],
                    "beat_grid": record["beat_grid"],
                    "valid": valid,
                    "sequences": all_sequences
                })

        window_offset += batch_size


assert window_offset == len(val_examples)


# ============================================================
# 5. Summarize progression behavior
# ============================================================

rows = []

for name, s in stats.items():

    f1 = (
        2 * s["tp"]
        /
        max(
            2 * s["tp"] + s["fp"] + s["fn"],
            1
        )
    )

    rows.append({
        "Model": name,

        "Avg unique chords":
            np.mean(s["unique"]),

        "Constant windows %":
            100 * s["constant"] / len(s["unique"]),

        "Chord change rate %":
            100 * s["changes"] / s["pairs"],

        "Avg longest repeated run":
            np.mean(s["longest"]),

        "I:MAJ prediction %":
            100 * s["tonic"] / s["beats"],

        "Implicit change F1":
            f1 if name != "Reference" else np.nan
    })


progression_comparison_df = pd.DataFrame(rows)

print("\nCHORD PROGRESSION QUALITY")
print("=" * 65)

display(
    progression_comparison_df.round(3)
)


# ============================================================
# 6. Most frequently predicted chords
# ============================================================

print("\nTOP 8 CHORDS BY MODEL")
print("=" * 65)

for name, s in stats.items():

    print(f"\n{name}")

    total = s["beats"]

    for chord, count in s["distribution"].most_common(8):

        print(
            f"  {chord_string(chord):12s} "
            f"{100 * count / total:6.2f}%"
        )


# ============================================================
# 7. Show actual 16-bar progressions
# ============================================================

print("\nTHREE VALIDATION PROGRESSION EXAMPLES")
print("=" * 65)

for example in sorted(
    progression_examples,
    key=lambda x: x["track"]
):

    print(
        f"\nTRACK: {example['track']} "
        f"| WINDOW: {example['window']}"
    )

    print("-" * 65)

    for name, sequence in example["sequences"].items():

        progression = bar_progression(
            sequence,
            example["valid"],
            example["beat_grid"]
        )

        print(f"\n{name}:")
        print(progression)


# ============================================================
# 8. Done
# ============================================================

print("\nEVALUATION COMPLETED")
print("--------------------------------")

print("Models compared:", len(models))
print("Validation windows:", len(val_examples))
print("Models retrained: 0")
print("Test partition accessed: NO")


CHORD PROGRESSION QUALITY


,Model,Avg unique chords,Constant windows %,Chord change rate %,Avg longest repeated run,I:MAJ prediction %,Implicit change F1
0,Reference,4.499,0.000,10.602,13.518,14.717,NaN
1,Model 1,4.737,5.409,16.720,13.643,15.632,0.141
2,Model 2,5.063,3.630,18.034,12.913,8.782,0.158
3,Model 2B Combined,4.979,1.851,18.521,12.547,14.163,0.155
4,Model 2B Chord,5.028,4.484,18.037,13.201,11.585,0.153



TOP 8 CHORDS BY MODEL

Reference
  VI:MIN        24.31%
  I:MAJ         14.72%
  IV:MAJ        13.81%
  V:MAJ         12.88%
  II:MIN        12.01%
  III:MIN        4.90%
  VI:MAJ         3.99%
  II:MAJ         3.13%

Model 1
  VI:MIN        33.14%
  I:MAJ         15.63%
  V:MAJ         14.13%
  II:MIN         9.12%
  IV:MAJ         6.60%
  II:MAJ         6.42%
  VI:MAJ         4.74%
  III:MIN        2.80%

Model 2
  VI:MIN        31.91%
  V:MAJ         12.43%
  IV:MAJ        10.25%
  II:MIN         9.69%
  I:MAJ          8.78%
  II:MAJ         6.55%
  VI:MAJ         4.11%
  ♭VII:MAJ       3.45%

Model 2B Combined
  VI:MIN        22.76%
  V:MAJ         18.34%
  I:MAJ         14.16%
  IV:MAJ        10.80%
  VI:MAJ         7.44%
  II:MAJ         6.11%
  II:MIN         5.78%
  ♭VII:MAJ       2.32%

Model 2B Chord
  VI:MIN        28.91%
  IV:MAJ        13.16%
  I:MAJ         11.59%
  V:MAJ         11.17%
  II:MIN        10.52%
  III:MIN        7.89%
  II:MAJ         4.95%
  IV:MIN        

In [12]:
# ============================================================
# MODEL 3 — HIERARCHICAL HARMONIC TRANSFORMER
#
# T / PD / D harmonic functions
# Boundary-guided harmonic state
# Root-conditioned chord-family prediction
# Song-frequency and class balancing
# Chord-consistency regularization
# ============================================================

import time
import math
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence


# ============================================================
# 1. Experiment configuration
# ============================================================

MODEL3_NAME = "phase3_v3_hierarchical_harmonic"

MODEL3_EPOCHS = 30
MODEL3_PATIENCE = 6

MODEL3_LR = 2e-4
MODEL3_DROPOUT = 0.20

FUNCTION_LOSS_WEIGHT = 0.5
BOUNDARY_LOSS_WEIGHT = 0.5
CONSISTENCY_LOSS_WEIGHT = 0.10

MODEL3_PATH = (
    CHECKPOINT_DIR
    / f"{MODEL3_NAME}_best_chord_loss.pt"
)

MODEL3_JOINT_PATH = (
    CHECKPOINT_DIR
    / f"{MODEL3_NAME}_best_joint_accuracy.pt"
)

FUNCTION_NAMES = ["T", "PD", "D"]

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# ============================================================
# 2. THEORY-DERIVED HARMONIC FUNCTION LABELS
# ============================================================

# Function classes:
# 0 = Tonic
# 1 = Predominant / Subdominant
# 2 = Dominant
#
# These are THEORY-DERIVED labels, not
# independently annotated function labels.
#
# Ambiguous or less conventional chords
# remain masked for function supervision.

FUNCTION_NAMES = ["T", "PD", "D"]

FUNCTION_MAP = {

    # ------------------------------------
    # TONIC FUNCTION
    # ------------------------------------

    "I:MAJ": 0,
    "I:MIN": 0,

    # Common tonic substitutes.
    # These can have other roles in context.

    "III:MIN": 0,
    "VI:MIN": 0,

    # Relative-major tonic in minor contexts.

    "BIII:MAJ": 0,


    # ------------------------------------
    # PREDOMINANT / SUBDOMINANT
    # ------------------------------------

    "II:MIN": 1,
    "II:DIM": 1,
    "II:HDIM": 1,

    "IV:MAJ": 1,
    "IV:MIN": 1,

    # Neapolitan and minor-key
    # predominant possibilities.

    "BII:MAJ": 1,
    "BVI:MAJ": 1,


    # ------------------------------------
    # DOMINANT FUNCTION
    # ------------------------------------

    "V:MAJ": 2,
    "V:DOM": 2,
    "V:SUS": 2,

    "VII:DIM": 2,
    "VII:HDIM": 2,
}


def normalize_function(value):

    if value is None:
        return IGNORE_INDEX

    if isinstance(value, dict):
        value = value.get(
            "plan_label",
            value.get("label")
        )

    if value is None:
        return IGNORE_INDEX

    label = str(value).strip().upper()

    # Normalize flat symbols:
    # ♭III:MAJ -> BIII:MAJ

    label = label.replace("♭", "B")
    label = label.replace(" ", "")

    return FUNCTION_MAP.get(
        label,
        IGNORE_INDEX
    )

# ============================================================
# 3. Hierarchical Dataset
# ============================================================

class HierarchicalHarmonyDataset(V3HarmonyDataset):

    def __init__(self, records):

        super().__init__(records)

        self.function_targets = []

        for record in records:

            labels = []

            for beat in record["beat_targets"]:

                # Only original reliable function labels.
                # Recovered chord labels are not assumed
                # to have reliable function annotations.

                original_valid = (
                    beat.get("model_target_mask") == 1
                )

                function = normalize_function(
                    beat.get("plan_label")
                )

                if not original_valid:
                    function = IGNORE_INDEX

                labels.append(function)

            self.function_targets.append(
                torch.tensor(
                    labels,
                    dtype=torch.long
                )
            )

    def __getitem__(self, index):

        item = super().__getitem__(index)

        item["function_target"] = (
            self.function_targets[index]
        )

        return item


def hierarchical_collate(samples):

    batch = v3_collate(samples)

    batch["function_target"] = pad_sequence(
        [
            item["function_target"]
            for item in samples
        ],
        batch_first=True,
        padding_value=IGNORE_INDEX
    )

    return batch


train_dataset3 = HierarchicalHarmonyDataset(
    train_examples
)

val_dataset3 = HierarchicalHarmonyDataset(
    val_examples
)


# ============================================================
# 4. Function class frequencies
# ============================================================

function_counts = torch.zeros(
    3,
    dtype=torch.long
)

for targets in train_dataset3.function_targets:

    valid = targets != IGNORE_INDEX

    if valid.any():

        function_counts += torch.bincount(
            targets[valid],
            minlength=3
        )


print("\nHARMONIC FUNCTION SUPERVISION")
print("--------------------------------")

for index, name in enumerate(FUNCTION_NAMES):

    print(
        name,
        ":",
        function_counts[index].item()
    )

assert (function_counts > 0).all(), (
    "Missing T/PD/D supervision. "
    "Check the existing beat_targets.plan_label values "
    "before training; do not invent function labels."
)

function_weights = calculate_weights(
    function_counts.float()
)


# ============================================================
# 5. Harmonic-change class balancing
# ============================================================

# Moderate positive weighting based on the
# actual V3 training distribution.

model3_change_pos_weight = torch.sqrt(
    change_counts[0].float()
    / change_counts[1].float()
).to(DEVICE)

print("\nCLASS BALANCING")
print("--------------------------------")

print(
    "Root weights:",
    root_weights.cpu().numpy().round(3)
)

print(
    "Family weights:",
    family_weights.cpu().numpy().round(3)
)

print(
    "Function weights:",
    function_weights.cpu().numpy().round(3)
)

print(
    "Change-positive weight:",
    round(
        model3_change_pos_weight.item(),
        4
    )
)


# ============================================================
# 6. Song-frequency balancing
# ============================================================

# Songs contribute different numbers of overlapping
# windows. Reduce the dominance of songs with many
# windows through moderate inverse-sqrt sampling.

track_window_counts = Counter(
    record["track_id"]
    for record in train_examples
)

sampling_weights = torch.tensor(
    [
        1.0 / math.sqrt(
            track_window_counts[
                record["track_id"]
            ]
        )
        for record in train_examples
    ],
    dtype=torch.double
)

sampling_generator = torch.Generator()

sampling_generator.manual_seed(SEED)

train_sampler3 = WeightedRandomSampler(
    weights=sampling_weights,
    num_samples=len(train_dataset3),
    replacement=True,
    generator=sampling_generator
)

train_loader3 = DataLoader(
    train_dataset3,
    batch_size=BATCH_SIZE,
    sampler=train_sampler3,
    collate_fn=hierarchical_collate
)

val_loader3 = DataLoader(
    val_dataset3,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hierarchical_collate
)

print("\nSONG BALANCING")
print("--------------------------------")

print(
    "Training songs:",
    len(track_window_counts)
)

print(
    "Training windows:",
    len(train_dataset3)
)

print(
    "Sampling:",
    "inverse-sqrt windows per song"
)

print(
    "Validation sampling:",
    "unchanged"
)


# ============================================================
# 7. Hierarchical Harmonic Architecture
# ============================================================

class HierarchicalHarmonicTransformer(
    HarmonyTemporalTransformer
):

    def __init__(self):

        super().__init__()

        # ------------------------------------
        # A. Regularization
        # ------------------------------------

        self.dropout = nn.Dropout(
            MODEL3_DROPOUT
        )

        # Rebuild Transformer layers with
        # increased dropout but same dimensions.

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=MODEL3_DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
            norm=nn.LayerNorm(D_MODEL),
            enable_nested_tensor=False
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=MODEL3_DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=2,
            norm=nn.LayerNorm(D_MODEL)
        )

        # ------------------------------------
        # B. Harmonic-function prediction
        # ------------------------------------

        self.function_head = nn.Linear(
            D_MODEL,
            3
        )

        self.function_condition = nn.Linear(
            3,
            D_MODEL,
            bias=False
        )

        # ------------------------------------
        # C. Harmonic-change prediction
        # ------------------------------------

        self.change_head = nn.Linear(
            D_MODEL,
            1
        )

        # ------------------------------------
        # D. Boundary-guided state
        # ------------------------------------

        self.state_input_norm = nn.LayerNorm(
            D_MODEL
        )

        self.state_output_norm = nn.LayerNorm(
            D_MODEL
        )

        # ------------------------------------
        # E. Root-conditioned family prediction
        # ------------------------------------

        self.root_condition = nn.Linear(
            12,
            D_MODEL,
            bias=False
        )

        self.family_condition_norm = nn.LayerNorm(
            D_MODEL
        )

    def forward(self, batch):

        # ====================================
        # STEP 1: Melody encoder
        # ====================================

        melody = (
            self.pitch_embedding(
                batch["melody_pitch"]
            )
            +
            self.melody_time_projection(
                batch["melody_time"]
            )
        )

        melody = add_positional_encoding(
            melody
        )

        melody = self.dropout(
            self.melody_norm(melody)
        )

        melody = melody.masked_fill(
            batch["melody_padding_mask"]
            .unsqueeze(-1),
            0.0
        )

        memory = self.encoder(
            melody,
            src_key_padding_mask=batch[
                "melody_padding_mask"
            ]
        )

        # ====================================
        # STEP 2: Temporal beat decoder
        # ====================================

        beat_features = (
            self.beat_time_projection(
                batch["beat_time"]
            )
            +
            self.bar_embedding(
                batch["bar_index"]
            )
            +
            self.beat_position_embedding(
                batch["beat_position"]
            )
            +
            self.meter_embedding(
                batch["meter"]
            )
        )

        beat_features = add_positional_encoding(
            beat_features
        )

        beat_features = self.dropout(
            self.beat_norm(beat_features)
        )

        beat_features = beat_features.masked_fill(
            batch["beat_padding_mask"]
            .unsqueeze(-1),
            0.0
        )

        temporal_bias = build_temporal_bias(
            batch
        )

        decoded = self.decoder(
            tgt=beat_features,
            memory=memory,
            memory_mask=temporal_bias,
            tgt_key_padding_mask=batch[
                "beat_padding_mask"
            ]
        )

        # ====================================
        # STEP 3: Harmonic function
        # ====================================

        function_logits = (
            self.function_head(decoded)
        )

        function_probs = F.softmax(
            function_logits,
            dim=-1
        )

        # Predicted T/PD/D distribution
        # conditions the chord representation.

        function_features = (
            self.function_condition(
                function_probs
            )
        )

        conditioned = self.state_input_norm(
            decoded
            +
            0.5 * function_features
        )

        # ====================================
        # STEP 4: Harmonic-change prediction
        # ====================================

        change_logits = (
            self.change_head(decoded)
            .squeeze(-1)
        )

        change_probs = torch.sigmoid(
            change_logits
        )

        # ====================================
        # STEP 5: Boundary-guided state
        # ====================================

        # Change labels identify events occurring
        # anywhere within a beat.
        #
        # Therefore, allow each predicted boundary
        # to influence both its own beat and the
        # following beat.

        previous_change = torch.cat(
            [
                torch.zeros_like(
                    change_probs[:, :1]
                ),
                change_probs[:, :-1]
            ],
            dim=1
        )

        effective_gate = (
            1.0
            -
            (1.0 - change_probs)
            *
            (1.0 - previous_change)
        )

        # Differentiable harmonic-state updates.

        states = []

        state = conditioned[:, 0, :]

        states.append(state)

        for t in range(
            1,
            conditioned.shape[1]
        ):

            gate = effective_gate[
                :, t
            ].unsqueeze(-1)

            state = (
                (1.0 - gate) * state
                +
                gate * conditioned[:, t, :]
            )

            states.append(state)

        harmonic_states = torch.stack(
            states,
            dim=1
        )

        # Small local residual preserves
        # melody information between boundaries.

        harmonic_states = (
            self.state_output_norm(
                harmonic_states
                +
                0.2 * conditioned
            )
        )

        # ====================================
        # STEP 6: Root prediction
        # ====================================

        root_logits = (
            self.root_head(
                harmonic_states
            )
        )

        root_probs = F.softmax(
            root_logits,
            dim=-1
        )

        # ====================================
        # STEP 7: Root-conditioned family
        # ====================================

        root_features = (
            self.root_condition(
                root_probs
            )
        )

        family_features = (
            self.family_condition_norm(
                harmonic_states
                +
                0.5 * root_features
            )
        )

        family_logits = (
            self.family_head(
                family_features
            )
        )

        return (
            root_logits,
            family_logits,
            function_logits,
            change_logits
        )


# ============================================================
# 8. Chord-consistency regularization
# ============================================================

def chord_consistency_loss(
    batch,
    root_logits,
    family_logits
):

    roots = batch["target_root"]
    families = batch["target_family"]

    valid = (
        (roots != IGNORE_INDEX)
        &
        (families != IGNORE_INDEX)
    )

    # Adjacent reliable beats containing
    # the same reference chord.

    same_chord = (
        valid[:, 1:]
        &
        valid[:, :-1]
        &
        (roots[:, 1:] == roots[:, :-1])
        &
        (families[:, 1:] == families[:, :-1])
    )

    if not same_chord.any():

        return root_logits.sum() * 0.0

    root_logp = F.log_softmax(
        root_logits,
        dim=-1
    )

    family_logp = F.log_softmax(
        family_logits,
        dim=-1
    )

    def symmetric_kl(logp):

        log_a = logp[:, :-1]
        log_b = logp[:, 1:]

        a = log_a.exp()
        b = log_b.exp()

        forward = (
            a * (log_a - log_b)
        ).sum(dim=-1)

        backward = (
            b * (log_b - log_a)
        ).sum(dim=-1)

        return 0.5 * (
            forward + backward
        )

    divergence = (
        symmetric_kl(root_logp)
        +
        symmetric_kl(family_logp)
    )

    return divergence[
        same_chord
    ].mean()


# ============================================================
# 9. Multi-task loss
# ============================================================

def model3_loss(batch, outputs):

    (
        root_logits,
        family_logits,
        function_logits,
        change_logits
    ) = outputs

    # ------------------------------------
    # A. Weighted chord classification
    # ------------------------------------

    root_loss = F.cross_entropy(
        root_logits.reshape(-1, 12),
        batch["target_root"].reshape(-1),
        weight=root_weights,
        ignore_index=IGNORE_INDEX
    )

    family_loss = F.cross_entropy(
        family_logits.reshape(-1, 7),
        batch["target_family"].reshape(-1),
        weight=family_weights,
        ignore_index=IGNORE_INDEX
    )

    # ------------------------------------
    # B. Weighted harmonic function
    # ------------------------------------

    function_loss = F.cross_entropy(
        function_logits.reshape(-1, 3),
        batch["function_target"].reshape(-1),
        weight=function_weights,
        ignore_index=IGNORE_INDEX
    )

    # ------------------------------------
    # C. Weighted harmonic changes
    # ------------------------------------

    change_targets = batch[
        "change_labels"
    ]

    change_valid = (
        change_targets != IGNORE_INDEX
    )

    if change_valid.any():

        change_loss = (
            F.binary_cross_entropy_with_logits(
                change_logits[change_valid],
                change_targets[
                    change_valid
                ].float(),
                pos_weight=model3_change_pos_weight
            )
        )

    else:

        change_loss = (
            change_logits.sum() * 0.0
        )

    # ------------------------------------
    # D. Chord consistency
    # ------------------------------------

    consistency_loss = (
        chord_consistency_loss(
            batch,
            root_logits,
            family_logits
        )
    )

    # ------------------------------------
    # E. Total loss
    # ------------------------------------

    total = (
        root_loss
        +
        family_loss
        +
        FUNCTION_LOSS_WEIGHT * function_loss
        +
        BOUNDARY_LOSS_WEIGHT * change_loss
        +
        CONSISTENCY_LOSS_WEIGHT * consistency_loss
    )

    return total


# ============================================================
# 10. Validation
# ============================================================

@torch.no_grad()
def validate_model3(model):

    model.eval()

    root_cm = torch.zeros(
        12, 12, dtype=torch.long
    )

    family_cm = torch.zeros(
        7, 7, dtype=torch.long
    )

    function_cm = torch.zeros(
        3, 3, dtype=torch.long
    )

    root_ce_sum = 0.0
    family_ce_sum = 0.0

    total_chords = 0
    joint_correct = 0

    boundary_tp = 0
    boundary_fp = 0
    boundary_fn = 0

    predicted_boundaries = 0
    total_boundary_beats = 0

    implicit_tp = 0
    implicit_fp = 0
    implicit_fn = 0

    for batch in val_loader3:

        batch = move_batch(batch)

        (
            root_logits,
            family_logits,
            function_logits,
            change_logits
        ) = model(batch)

        # ------------------------------------
        # Chord accuracy
        # ------------------------------------

        true_root = batch["target_root"]
        true_family = batch["target_family"]

        valid = (
            (true_root != IGNORE_INDEX)
            &
            (true_family != IGNORE_INDEX)
        )

        n = int(
            valid.sum().item()
        )

        total_chords += n

        root_ce_sum += F.cross_entropy(
            root_logits.reshape(-1, 12),
            true_root.reshape(-1),
            ignore_index=IGNORE_INDEX,
            reduction="sum"
        ).item()

        family_ce_sum += F.cross_entropy(
            family_logits.reshape(-1, 7),
            true_family.reshape(-1),
            ignore_index=IGNORE_INDEX,
            reduction="sum"
        ).item()

        predicted_root = (
            root_logits.argmax(-1)
        )

        predicted_family = (
            family_logits.argmax(-1)
        )

        root_cm += torch.bincount(
            (
                true_root[valid] * 12
                +
                predicted_root[valid]
            ).cpu(),
            minlength=144
        ).reshape(12, 12)

        family_cm += torch.bincount(
            (
                true_family[valid] * 7
                +
                predicted_family[valid]
            ).cpu(),
            minlength=49
        ).reshape(7, 7)

        joint_correct += int(
            (
                valid
                &
                (predicted_root == true_root)
                &
                (predicted_family == true_family)
            ).sum().item()
        )

        # ------------------------------------
        # Harmonic-function evaluation
        # ------------------------------------

        true_function = batch[
            "function_target"
        ]

        function_valid = (
            true_function != IGNORE_INDEX
        )

        if function_valid.any():

            predicted_function = (
                function_logits.argmax(-1)
            )

            function_cm += torch.bincount(
                (
                    true_function[function_valid] * 3
                    +
                    predicted_function[function_valid]
                ).cpu(),
                minlength=9
            ).reshape(3, 3)

        # ------------------------------------
        # Event-based change evaluation
        # ------------------------------------

        true_change = batch[
            "change_labels"
        ]

        change_valid = (
            true_change != IGNORE_INDEX
        )

        if change_valid.any():

            predicted_change = (
                torch.sigmoid(
                    change_logits[change_valid]
                ) >= 0.5
            )

            actual_change = (
                true_change[change_valid] == 1
            )

            boundary_tp += int(
                (
                    predicted_change
                    &
                    actual_change
                ).sum().item()
            )

            boundary_fp += int(
                (
                    predicted_change
                    &
                    ~actual_change
                ).sum().item()
            )

            boundary_fn += int(
                (
                    ~predicted_change
                    &
                    actual_change
                ).sum().item()
            )

            predicted_boundaries += int(
                predicted_change.sum().item()
            )

            total_boundary_beats += int(
                change_valid.sum().item()
            )

        # ------------------------------------
        # Implicit chord-change evaluation
        # ------------------------------------

        pair_valid = (
            valid[:, 1:]
            &
            valid[:, :-1]
        )

        actual_pair_change = (
            (true_root[:, 1:] != true_root[:, :-1])
            |
            (true_family[:, 1:] != true_family[:, :-1])
        )

        predicted_pair_change = (
            (predicted_root[:, 1:] != predicted_root[:, :-1])
            |
            (predicted_family[:, 1:] != predicted_family[:, :-1])
        )

        implicit_tp += int(
            (
                pair_valid
                &
                actual_pair_change
                &
                predicted_pair_change
            ).sum().item()
        )

        implicit_fp += int(
            (
                pair_valid
                &
                ~actual_pair_change
                &
                predicted_pair_change
            ).sum().item()
        )

        implicit_fn += int(
            (
                pair_valid
                &
                actual_pair_change
                &
                ~predicted_pair_change
            ).sum().item()
        )

    # ----------------------------------------
    # Final metrics
    # ----------------------------------------

    function_total = (
        function_cm.sum().item()
    )

    return {

        "val_chord_ce": (
            root_ce_sum + family_ce_sum
        ) / total_chords,

        "root_acc":
            root_cm.diag().sum().item()
            / total_chords,

        "family_acc":
            family_cm.diag().sum().item()
            / total_chords,

        "joint_acc":
            joint_correct / total_chords,

        "root_f1":
            macro_f1(root_cm),

        "family_f1":
            macro_f1(family_cm),

        "function_acc":
            function_cm.diag().sum().item()
            / max(function_total, 1),

        "function_f1":
            macro_f1(function_cm),

        "event_change_f1": (
            2 * boundary_tp
            /
            max(
                2 * boundary_tp
                + boundary_fp
                + boundary_fn,
                1
            )
        ),

        "event_change_precision": (
            boundary_tp
            / max(
                boundary_tp + boundary_fp,
                1
            )
        ),

        "event_change_recall": (
            boundary_tp
            / max(
                boundary_tp + boundary_fn,
                1
            )
        ),

        "predicted_change_rate": (
            predicted_boundaries
            / max(total_boundary_beats, 1)
        ),

        "implicit_change_f1": (
            2 * implicit_tp
            /
            max(
                2 * implicit_tp
                + implicit_fp
                + implicit_fn,
                1
            )
        )
    }


# ============================================================
# 11. Initialize Model 3
# ============================================================

model3 = (
    HierarchicalHarmonicTransformer()
    .to(DEVICE)
)

model3_parameters = sum(
    parameter.numel()
    for parameter in model3.parameters()
    if parameter.requires_grad
)

optimizer3 = torch.optim.AdamW(
    model3.parameters(),
    lr=MODEL3_LR,
    weight_decay=WEIGHT_DECAY
)

scheduler3 = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer3,
        T_max=MODEL3_EPOCHS,
        eta_min=1e-5
    )
)


# ============================================================
# 12. Training
# ============================================================

history3 = []

best_chord_ce = float("inf")
best_joint_acc = -1.0

best_chord_epoch = None
best_joint_epoch = None

epochs_without_improvement = 0


print("\n" + "=" * 60)
print("MODEL 3 — HIERARCHICAL HARMONIC TRANSFORMER")
print("=" * 60)

print("Device:", DEVICE)
print("Training windows:", len(train_dataset3))
print("Validation windows:", len(val_dataset3))

print("Parameters:", f"{model3_parameters:,}")

print("Encoder layers:", 2)
print("Decoder layers:", 2)

print("Maximum epochs:", MODEL3_EPOCHS)
print("Early stopping patience:", MODEL3_PATIENCE)

print("Dropout:", MODEL3_DROPOUT)

print("Function loss weight:", FUNCTION_LOSS_WEIGHT)
print("Boundary loss weight:", BOUNDARY_LOSS_WEIGHT)
print("Consistency weight:", CONSISTENCY_LOSS_WEIGHT)

print("\nTraining started...\n")


for epoch in range(1, MODEL3_EPOCHS + 1):

    start_time = time.perf_counter()

    model3.train()

    train_loss_sum = 0.0

    for batch in train_loader3:

        batch = move_batch(batch)

        optimizer3.zero_grad(
            set_to_none=True
        )

        outputs = model3(batch)

        loss = model3_loss(
            batch,
            outputs
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model3.parameters(),
            GRAD_CLIP
        )

        optimizer3.step()

        train_loss_sum += loss.item()

    train_loss = (
        train_loss_sum / len(train_loader3)
    )

    metrics = validate_model3(
        model3
    )

    current_lr = (
        optimizer3.param_groups[0]["lr"]
    )

    scheduler3.step()

    history3.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "lr": current_lr,
        **metrics
    })

    # ----------------------------------------
    # A. Best validation chord loss
    # ----------------------------------------

    improved_chord = (
        metrics["val_chord_ce"]
        < best_chord_ce - 1e-4
    )

    if improved_chord:

        best_chord_ce = metrics[
            "val_chord_ce"
        ]

        best_chord_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model3.state_dict(),
                "validation_metrics": metrics,
                "experiment": MODEL3_NAME,
                "dataset": DATASET_PATH.name,
                "parameter_count": model3_parameters,
                "function_names": FUNCTION_NAMES,
                "seed": SEED
            },
            MODEL3_PATH
        )

    else:

        epochs_without_improvement += 1

    # ----------------------------------------
    # B. Best exact chord accuracy
    # ----------------------------------------

    improved_joint = (
        metrics["joint_acc"]
        > best_joint_acc
    )

    if improved_joint:

        best_joint_acc = metrics[
            "joint_acc"
        ]

        best_joint_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model3.state_dict(),
                "validation_metrics": metrics,
                "experiment": MODEL3_NAME,
                "dataset": DATASET_PATH.name,
                "parameter_count": model3_parameters,
                "seed": SEED
            },
            MODEL3_JOINT_PATH
        )

    elapsed = (
        time.perf_counter() - start_time
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train: {train_loss:.4f} | "
        f"Val CE: {metrics['val_chord_ce']:.4f} | "
        f"Joint: {metrics['joint_acc']:.2%} | "
        f"Function: {metrics['function_acc']:.2%} | "
        f"Function F1: {metrics['function_f1']:.4f} | "
        f"Event F1: {metrics['event_change_f1']:.4f} | "
        f"Implicit F1: {metrics['implicit_change_f1']:.4f} | "
        f"{elapsed:.1f}s"
        + (" | BEST CE" if improved_chord else "")
        + (" | BEST JOINT" if improved_joint else "")
    )

    if epochs_without_improvement >= MODEL3_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break


# ============================================================
# 13. Evaluate saved checkpoints
# ============================================================

model3_history_df = pd.DataFrame(
    history3
)

results3 = []

for checkpoint_name, path in [
    ("Best chord loss", MODEL3_PATH),
    ("Best joint accuracy", MODEL3_JOINT_PATH)
]:

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
        weights_only=True
    )

    model3.load_state_dict(
        checkpoint["model_state_dict"]
    )

    metrics = validate_model3(
        model3
    )

    results3.append({
        "Checkpoint": checkpoint_name,
        "Epoch": checkpoint["epoch"],
        "Joint accuracy": metrics["joint_acc"],
        "Root accuracy": metrics["root_acc"],
        "Family accuracy": metrics["family_acc"],
        "Root macro-F1": metrics["root_f1"],
        "Function accuracy": metrics["function_acc"],
        "Function macro-F1": metrics["function_f1"],
        "Event-change F1": metrics["event_change_f1"],
        "Implicit-change F1": metrics["implicit_change_f1"],
        "Predicted change rate":
            metrics["predicted_change_rate"]
    })

model3_results_df = pd.DataFrame(
    results3
)

print("\n" + "=" * 60)
print("MODEL 3 — FINAL RESULTS")
print("=" * 60)

display(
    model3_results_df.round(4)
)


# ============================================================
# 14. Compare with earlier chord models
# ============================================================

print("\nCHORD ACCURACY COMPARISON")
print("--------------------------------")

comparison3 = pd.DataFrame([
    {
        "Model": "V2 Temporal",
        "Joint accuracy": 0.3096
    },
    {
        "Model": "V3 Model 1",
        "Joint accuracy": 0.2829
    },
    {
        "Model": "V3 Model 2",
        "Joint accuracy": 0.2712
    },
    {
        "Model": "V3 Model 2B — Best chord",
        "Joint accuracy": 0.2803
    },
    {
        "Model": "V3 Model 3 — Best chord loss",
        "Joint accuracy":
            results3[0]["Joint accuracy"]
    },
    {
        "Model": "V3 Model 3 — Best joint",
        "Joint accuracy":
            results3[1]["Joint accuracy"]
    }
])

display(
    comparison3.round(4)
)


print("\nTRAINING COMPLETE")
print("--------------------------------")

print("Best chord-loss epoch:", best_chord_epoch)
print("Best joint-accuracy epoch:", best_joint_epoch)

print("Training windows:", len(train_dataset3))
print("Validation windows:", len(val_dataset3))

print("Checkpoints saved:", 2)

print("Original dataset modified: NO")
print("Test partition accessed: NO")


HARMONIC FUNCTION SUPERVISION
--------------------------------
T : 104507
PD : 65346
D : 38770

CLASS BALANCING
--------------------------------
Root weights: [0.833 4.    1.007 4.    1.218 0.839 4.    0.859 4.    0.732 2.8   3.566]
Family weights: [0.765 0.952 2.888 4.    4.    4.    4.   ]
Function weights: [0.832 1.052 1.366]
Change-positive weight: 1.9113

SONG BALANCING
--------------------------------
Training songs: 701
Training windows: 4523
Sampling: inverse-sqrt windows per song
Validation sampling: unchanged

MODEL 3 — HIERARCHICAL HARMONIC TRANSFORMER
Device: cpu
Training windows: 4523
Validation windows: 1405
Parameters: 954,135
Encoder layers: 2
Decoder layers: 2
Maximum epochs: 30
Early stopping patience: 6
Dropout: 0.2
Function loss weight: 0.5
Boundary loss weight: 0.5
Consistency weight: 0.1

Training started...

Epoch 01 | Train: 4.0954 | Val CE: 2.6741 | Joint: 25.36% | Function: 52.91% | Function F1: 0.4790 | Event F1: 0.0366 | Implicit F1: 0.1456 | 18.0s | BEST C

,Checkpoint,Epoch,Joint accuracy,Root accuracy,Family accuracy,Root macro-F1,Function accuracy,Function macro-F1,Event-change F1,Implicit-change F1,Predicted change rate
0,Best chord loss,5,0.2722,0.3520,0.5836,0.1901,0.5259,0.4736,0.0573,0.1503,0.0188
1,Best joint accuracy,2,0.2739,0.3574,0.5726,0.1790,0.5528,0.4849,0.0664,0.1460,0.0225



CHORD ACCURACY COMPARISON
--------------------------------


,Model,Joint accuracy
0,V2 Temporal,0.3096
1,V3 Model 1,0.2829
2,V3 Model 2,0.2712
3,V3 Model 2B — Best chord,0.2803
4,V3 Model 3 — Best chord loss,0.2722
5,V3 Model 3 — Best joint,0.2739



TRAINING COMPLETE
--------------------------------
Best chord-loss epoch: 5
Best joint-accuracy epoch: 2
Training windows: 4523
Validation windows: 1405
Checkpoints saved: 2
Original dataset modified: NO
Test partition accessed: NO


In [11]:
# CELL 6A — Identify existing harmonic-function labels

from collections import Counter

label_counts = Counter()
label_examples = {}

for window in train_examples:

    for beat in window["beat_targets"]:

        if beat.get("model_target_mask") != 1:
            continue

        label = repr(beat.get("plan_label"))

        label_counts[label] += 1

        if label not in label_examples:
            label_examples[label] = (
                beat.get("root_id"),
                beat.get("family_id")
            )

print("PLAN LABEL DISTRIBUTION")
print("-" * 50)

print("Reliable beats:", sum(label_counts.values()))
print("Unique labels:", len(label_counts))

for label, count in label_counts.most_common(25):

    print(
        f"{label:25s} "
        f"Count: {count:7d} "
        f"Example root/family: {label_examples[label]}"
    )

print("\nSAMPLE ORIGINAL BEAT TARGET")
print("-" * 50)

print(train_examples[0]["beat_targets"][0])

PLAN LABEL DISTRIBUTION
--------------------------------------------------
Reliable beats: 239602
Unique labels: 67
'VI:MIN'                  Count:   49558 Example root/family: (9, 1)
'IV:MAJ'                  Count:   40973 Example root/family: (5, 0)
'I:MAJ'                   Count:   40465 Example root/family: (0, 0)
'V:MAJ'                   Count:   34649 Example root/family: (7, 0)
'II:MIN'                  Count:   22361 Example root/family: (2, 1)
'III:MIN'                 Count:   12319 Example root/family: (4, 1)
'III:MAJ'                 Count:    6254 Example root/family: (4, 0)
'II:MAJ'                  Count:    5128 Example root/family: (2, 0)
'VI:MAJ'                  Count:    4711 Example root/family: (9, 0)
'V:DOM'                   Count:    3477 Example root/family: (7, 2)
'♭VII:MAJ'                Count:    3408 Example root/family: (10, 0)
'V:MIN'                   Count:    1666 Example root/family: (7, 1)
'II:DOM'                  Count:    1488 Example root/f

### Class Transformer + RL Fine-Tuning

In [13]:
# ============================================================
# CELL 7 — FLAT 84-CLASS TRANSFORMER + RL
#
# Stage A: Initialize from pretrained Model 1
# Stage B: Supervised flat-chord training
# Stage C: Reinforcement learning (SCST / REINFORCE)
# Stage D: Compare before and after RL
# ============================================================

import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.distributions import Categorical


# ============================================================
# 1. Settings
# ============================================================

FLAT_CLASSES = 12 * 7

SUPERVISED_EPOCHS = 5
RL_EPOCHS = 10

SUPERVISED_LR = 1e-4
RL_LR = 3e-5

RL_CE_WEIGHT = 0.7
RL_POLICY_WEIGHT = 3.0

torch.manual_seed(SEED)

FLAT_PRETRAIN_PATH = (
    CHECKPOINT_DIR / "flat84_before_rl.pt"
)

FLAT_RL_PATH = (
    CHECKPOINT_DIR / "flat84_after_rl_best.pt"
)


# ============================================================
# 2. Flat 84-class Transformer
# ============================================================

class Flat84Transformer(HarmonyTemporalTransformer):

    def __init__(self, pretrained_model):

        super().__init__()

        # Copy the already-trained temporal backbone.

        self.load_state_dict(
            pretrained_model.state_dict()
        )

        self.flat_head = nn.Linear(
            D_MODEL, FLAT_CLASSES
        )

        # Initialize the flat head using the
        # pretrained root and family heads.

        root_indices = (
            torch.arange(FLAT_CLASSES) // 7
        )

        family_indices = (
            torch.arange(FLAT_CLASSES) % 7
        )

        with torch.no_grad():

            self.flat_head.weight.copy_(
                self.root_head.weight[root_indices]
                +
                self.family_head.weight[family_indices]
            )

            self.flat_head.bias.copy_(
                self.root_head.bias[root_indices]
                +
                self.family_head.bias[family_indices]
            )

        del self.root_head
        del self.family_head

    def forward(self, batch):

        # Melody representation.

        melody = (
            self.pitch_embedding(batch["melody_pitch"])
            +
            self.melody_time_projection(batch["melody_time"])
        )

        melody = add_positional_encoding(melody)

        melody = self.dropout(
            self.melody_norm(melody)
        )

        melody = melody.masked_fill(
            batch["melody_padding_mask"].unsqueeze(-1),
            0.0
        )

        memory = self.encoder(
            melody,
            src_key_padding_mask=batch[
                "melody_padding_mask"
            ]
        )

        # Beat representation.

        beats = (
            self.beat_time_projection(batch["beat_time"])
            +
            self.bar_embedding(batch["bar_index"])
            +
            self.beat_position_embedding(batch["beat_position"])
            +
            self.meter_embedding(batch["meter"])
        )

        beats = add_positional_encoding(beats)

        beats = self.dropout(
            self.beat_norm(beats)
        )

        beats = beats.masked_fill(
            batch["beat_padding_mask"].unsqueeze(-1),
            0.0
        )

        decoded = self.decoder(
            tgt=beats,
            memory=memory,
            memory_mask=build_temporal_bias(batch),
            tgt_key_padding_mask=batch["beat_padding_mask"]
        )

        # One joint distribution over all 84 chords.

        return self.flat_head(decoded)


flat_model = Flat84Transformer(model1).to(DEVICE)

print("FLAT 84-CLASS TRANSFORMER")
print("--------------------------------")
print(
    "Parameters:",
    f"{sum(p.numel() for p in flat_model.parameters()):,}"
)


# ============================================================
# 3. Prepare flat labels
# ============================================================

def flat_targets(batch):

    roots = batch["target_root"]
    families = batch["target_family"]

    valid = (
        (roots != IGNORE_INDEX)
        &
        (families != IGNORE_INDEX)
    )

    targets = (
        roots.clamp(min=0) * 7
        +
        families.clamp(min=0)
    )

    return targets.masked_fill(
        ~valid,
        IGNORE_INDEX
    )


# ============================================================
# 4. Balanced flat-class weights
# ============================================================

flat_counts = torch.zeros(FLAT_CLASSES)

for record in train_examples:

    targets = record["v3_targets"]

    for root, family, mask in zip(
        targets["root_ids"],
        targets["family_ids"],
        targets["chord_mask"]
    ):

        if mask:
            flat_counts[root * 7 + family] += 1

present = flat_counts > 0

flat_weights = torch.ones(FLAT_CLASSES)

median_count = flat_counts[present].median()

# Tempered inverse-frequency weighting.
# Less aggressive than full inverse-frequency.

flat_weights[present] = (
    median_count / flat_counts[present]
).pow(0.25).clamp(
    min=0.7,
    max=2.5
)

flat_weights = flat_weights.to(DEVICE)

print("\nCLASS BALANCING")
print("--------------------------------")
print("Observed chord classes:", int(present.sum()))
print("Total classes:", FLAT_CLASSES)
print("Weight range: 0.7–2.5")


def flat_ce_loss(logits, targets):

    return F.cross_entropy(
        logits.reshape(-1, FLAT_CLASSES),
        targets.reshape(-1),
        weight=flat_weights,
        ignore_index=IGNORE_INDEX
    )


# ============================================================
# 5. Sequence reward
# ============================================================

# Rewards are calculated for complete windows.
#
# 55%: exact chord correctness
# 30%: chord-change F1
# 15%: harmonic-rhythm similarity
#
# A small penalty discourages constant-chord
# predictions when the reference is varied.
#
# Masked chord targets are excluded.
# This is a reference-based reward, not an
# independent music-quality judgment.


@torch.no_grad()
def sequence_rewards(actions, targets):

    B, T = actions.shape

    rewards = []

    for b in range(B):

        reference = targets[b]
        predicted = actions[b]

        valid = reference != IGNORE_INDEX

        if not valid.any():

            rewards.append(0.0)
            continue

        # ------------------------------------
        # A. Chord accuracy
        # ------------------------------------

        accuracy = (
            predicted[valid] == reference[valid]
        ).float().mean().item()

        # ------------------------------------
        # B. Adjacent reliable transitions
        # ------------------------------------

        pair_valid = (
            valid[1:] & valid[:-1]
        )

        true_changes = (
            reference[1:] != reference[:-1]
        )[pair_valid]

        pred_changes = (
            predicted[1:] != predicted[:-1]
        )[pair_valid]

        if len(true_changes) > 0:

            tp = (
                true_changes & pred_changes
            ).sum().item()

            fp = (
                ~true_changes & pred_changes
            ).sum().item()

            fn = (
                true_changes & ~pred_changes
            ).sum().item()

            if tp + fp + fn == 0:
                change_f1 = 1.0
            else:
                change_f1 = (
                    2 * tp
                    /
                    (2 * tp + fp + fn)
                )

            actual_rate = (
                true_changes.float().mean().item()
            )

            predicted_rate = (
                pred_changes.float().mean().item()
            )

            rhythm_score = (
                1.0
                -
                abs(actual_rate - predicted_rate)
            )

        else:

            change_f1 = 0.0
            rhythm_score = 0.0

        # ------------------------------------
        # C. Diversity penalty
        # ------------------------------------

        true_unique = (
            torch.unique(reference[valid]).numel()
        )

        predicted_unique = (
            torch.unique(predicted[valid]).numel()
        )

        collapse_penalty = (
            0.05
            if true_unique >= 3
            and predicted_unique == 1
            else 0.0
        )

        # ------------------------------------
        # D. Total sequence reward
        # ------------------------------------

        reward = (
            0.55 * accuracy
            +
            0.30 * change_f1
            +
            0.15 * rhythm_score
            -
            collapse_penalty
        )

        rewards.append(reward)

    return torch.tensor(
        rewards,
        dtype=torch.float32,
        device=actions.device
    )


# ============================================================
# 6. Evaluation
# ============================================================

@torch.no_grad()
def evaluate_flat(model):

    model.eval()

    correct = 0
    total = 0

    tp = 0
    fp = 0
    fn = 0

    pred_changes = 0
    valid_pairs = 0

    unique_counts = []
    constant_windows = 0

    for batch in val_loader:

        batch = move_batch(batch)

        targets = flat_targets(batch)

        logits = model(batch)

        predictions = logits.argmax(-1)

        valid = targets != IGNORE_INDEX

        correct += int(
            (
                (predictions == targets)
                & valid
            ).sum().item()
        )

        total += int(valid.sum().item())

        for b in range(predictions.size(0)):

            mask = valid[b]

            if not mask.any():
                continue

            unique = torch.unique(
                predictions[b][mask]
            ).numel()

            unique_counts.append(unique)

            constant_windows += int(unique == 1)

        pair_valid = (
            valid[:, 1:] & valid[:, :-1]
        )

        actual = (
            targets[:, 1:] != targets[:, :-1]
        )

        predicted = (
            predictions[:, 1:] != predictions[:, :-1]
        )

        tp += int(
            (
                predicted & actual & pair_valid
            ).sum().item()
        )

        fp += int(
            (
                predicted & ~actual & pair_valid
            ).sum().item()
        )

        fn += int(
            (
                ~predicted & actual & pair_valid
            ).sum().item()
        )

        pred_changes += int(
            (
                predicted & pair_valid
            ).sum().item()
        )

        valid_pairs += int(
            pair_valid.sum().item()
        )

    joint_accuracy = correct / max(total, 1)

    change_f1 = (
        2 * tp
        /
        max(2 * tp + fp + fn, 1)
    )

    change_rate = (
        pred_changes / max(valid_pairs, 1)
    )

    return {
        "joint_accuracy": joint_accuracy,
        "change_f1": change_f1,
        "change_rate": change_rate,
        "unique_chords": float(np.mean(unique_counts)),
        "constant_windows": (
            constant_windows / len(unique_counts)
        )
    }


def selection_score(metrics):

    # Experimental progression-quality selection.
    # Chord accuracy remains the larger component.

    return (
        0.7 * metrics["joint_accuracy"]
        +
        0.3 * metrics["change_f1"]
    )


# ============================================================
# 7. Stage A — Supervised flat-chord training
# ============================================================

print("\n" + "=" * 60)
print("STAGE A — SUPERVISED FLAT-CHORD TRAINING")
print("=" * 60)

optimizer_flat = torch.optim.AdamW(
    flat_model.parameters(),
    lr=SUPERVISED_LR,
    weight_decay=WEIGHT_DECAY
)

best_supervised_score = -1.0

for epoch in range(1, SUPERVISED_EPOCHS + 1):

    flat_model.train()

    total_loss = 0.0

    start = time.perf_counter()

    # Reuse song-balanced training loader.

    for batch in train_loader3:

        batch = move_batch(batch)

        targets = flat_targets(batch)

        optimizer_flat.zero_grad(
            set_to_none=True
        )

        logits = flat_model(batch)

        loss = flat_ce_loss(
            logits,
            targets
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            flat_model.parameters(),
            GRAD_CLIP
        )

        optimizer_flat.step()

        total_loss += loss.item()

    metrics = evaluate_flat(flat_model)

    score = selection_score(metrics)

    if score > best_supervised_score:

        best_supervised_score = score

        torch.save(
            {
                "model_state_dict":
                    flat_model.state_dict(),
                "epoch": epoch,
                "metrics": metrics,
                "stage": "supervised"
            },
            FLAT_PRETRAIN_PATH
        )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {total_loss / len(train_loader3):.4f} | "
        f"Joint: {metrics['joint_accuracy']:.2%} | "
        f"Change F1: {metrics['change_f1']:.4f} | "
        f"Change rate: {metrics['change_rate']:.2%} | "
        f"{time.perf_counter()-start:.1f}s"
    )


# Restore best supervised checkpoint.

checkpoint = torch.load(
    FLAT_PRETRAIN_PATH,
    map_location=DEVICE,
    weights_only=True
)

flat_model.load_state_dict(
    checkpoint["model_state_dict"]
)

before_rl = evaluate_flat(flat_model)


# ============================================================
# 8. Stage B — Reinforcement learning
# ============================================================

print("\n" + "=" * 60)
print("STAGE B — SELF-CRITICAL REINFORCEMENT LEARNING")
print("=" * 60)

print("RL algorithm: Self-Critical Policy Gradient")
print("Supervised loss coefficient:", RL_CE_WEIGHT)
print("Policy loss coefficient:", RL_POLICY_WEIGHT)
print("RL epochs:", RL_EPOCHS)

optimizer_rl = torch.optim.AdamW(
    flat_model.parameters(),
    lr=RL_LR,
    weight_decay=WEIGHT_DECAY
)

best_rl_score = selection_score(before_rl)

# The pre-RL checkpoint is our fallback
# if RL never improves the selected metric.

best_rl_state = copy.deepcopy(
    flat_model.state_dict()
)

best_rl_epoch = 0

rl_history = []

for epoch in range(1, RL_EPOCHS + 1):

    start = time.perf_counter()

    flat_model.train()

    total_loss = 0.0
    total_rl_loss = 0.0
    total_advantage = 0.0

    for batch in train_loader3:

        batch = move_batch(batch)

        targets = flat_targets(batch)

        valid = targets != IGNORE_INDEX

        optimizer_rl.zero_grad(
            set_to_none=True
        )

        logits = flat_model(batch)

        distribution = Categorical(
            logits=logits
        )

        # Sample a complete chord progression.

        sampled_actions = (
            distribution.sample()
        )

        # Greedy progression serves as baseline.

        greedy_actions = (
            logits.argmax(-1)
        )

        # ------------------------------------
        # A. Sequence-level rewards
        # ------------------------------------

        sampled_reward = sequence_rewards(
            sampled_actions,
            targets
        )

        greedy_reward = sequence_rewards(
            greedy_actions,
            targets
        )

        # Self-critical advantage.

        advantage = (
            sampled_reward - greedy_reward
        ).detach()

        # ------------------------------------
        # B. Policy-gradient loss
        # ------------------------------------

        action_log_probs = (
            distribution.log_prob(
                sampled_actions
            )
        )

        # Only reliable positions contribute
        # to the current reference-based reward.

        sequence_log_prob = (
            (action_log_probs * valid.float())
            .sum(dim=1)
            /
            valid.sum(dim=1).clamp(min=1)
        )

        rl_loss = -(
            advantage
            *
            sequence_log_prob
        ).mean()

        # ------------------------------------
        # C. Supervised anchor
        # ------------------------------------

        ce_loss = flat_ce_loss(
            logits,
            targets
        )

        # Keeps the policy from drifting too far
        # from supervised chord prediction.

        loss = (
            RL_CE_WEIGHT * ce_loss
            +
            RL_POLICY_WEIGHT * rl_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            flat_model.parameters(),
            GRAD_CLIP
        )

        optimizer_rl.step()

        total_loss += loss.item()
        total_rl_loss += rl_loss.item()

        total_advantage += (
            advantage.mean().item()
        )

    # ----------------------------------------
    # D. Validation
    # ----------------------------------------

    metrics = evaluate_flat(flat_model)

    score = selection_score(metrics)

    rl_history.append({
        "epoch": epoch,
        "train_loss": total_loss / len(train_loader3),
        "policy_loss": total_rl_loss / len(train_loader3),
        "advantage": total_advantage / len(train_loader3),
        **metrics
    })

    if score > best_rl_score:

        best_rl_score = score
        best_rl_epoch = epoch

        best_rl_state = copy.deepcopy(
            flat_model.state_dict()
        )

        torch.save(
            {
                "model_state_dict":
                    best_rl_state,
                "epoch": epoch,
                "metrics": metrics,
                "stage": "reinforcement_learning",
                "reward_weights": {
                    "accuracy": 0.55,
                    "change_f1": 0.30,
                    "rhythm": 0.15
                }
            },
            FLAT_RL_PATH
        )

    print(
        f"RL Epoch {epoch:02d} | "
        f"Train: {total_loss / len(train_loader3):.4f} | "
        f"Policy: {total_rl_loss / len(train_loader3):+.4f} | "
        f"Joint: {metrics['joint_accuracy']:.2%} | "
        f"Change F1: {metrics['change_f1']:.4f} | "
        f"Change rate: {metrics['change_rate']:.2%} | "
        f"Unique: {metrics['unique_chords']:.2f} | "
        f"{time.perf_counter()-start:.1f}s"
        + (" | BEST" if best_rl_epoch == epoch else "")
    )


# ============================================================
# 9. Final before/after comparison
# ============================================================

flat_model.load_state_dict(
    best_rl_state
)

after_rl = evaluate_flat(flat_model)

print("\n" + "=" * 60)
print("FLAT TRANSFORMER — BEFORE VS AFTER RL")
print("=" * 60)

comparison_rl = pd.DataFrame([
    {
        "Model": "Supervised Flat 84",
        **before_rl
    },
    {
        "Model": "Flat 84 + RL",
        **after_rl
    }
])

display(
    comparison_rl.round(4)
)

print("\nBest RL epoch:", best_rl_epoch)

if best_rl_epoch == 0:

    print(
        "RL did not improve the selected validation score."
    )

else:

    print(
        "RL improved the selected validation score."
    )

print("\nOriginal V2 Flat baseline: 31.60%")
print("Test partition accessed: NO")
print("Experiment complete.")

FLAT 84-CLASS TRANSFORMER
--------------------------------
Parameters: 959,316

CLASS BALANCING
--------------------------------
Observed chord classes: 67
Total classes: 84
Weight range: 0.7–2.5

STAGE A — SUPERVISED FLAT-CHORD TRAINING
Epoch 01 | Loss: 2.2399 | Joint: 33.61% | Change F1: 0.1307 | Change rate: 11.01% | 16.6s
Epoch 02 | Loss: 2.1188 | Joint: 32.53% | Change F1: 0.1320 | Change rate: 12.24% | 16.1s
Epoch 03 | Loss: 2.1046 | Joint: 33.95% | Change F1: 0.1280 | Change rate: 11.41% | 15.1s
Epoch 04 | Loss: 2.0847 | Joint: 33.85% | Change F1: 0.1291 | Change rate: 11.14% | 16.2s
Epoch 05 | Loss: 2.0674 | Joint: 33.67% | Change F1: 0.1319 | Change rate: 11.20% | 16.1s

STAGE B — SELF-CRITICAL REINFORCEMENT LEARNING
RL algorithm: Self-Critical Policy Gradient
Supervised loss coefficient: 0.7
Policy loss coefficient: 3.0
RL epochs: 10
RL Epoch 01 | Train: 0.6896 | Policy: -0.2574 | Joint: 34.18% | Change F1: 0.1241 | Change rate: 11.12% | Unique: 3.25 | 17.1s | BEST
RL Epoch 0

,Model,joint_accuracy,change_f1,change_rate,unique_chords,constant_windows
0,Supervised Flat 84,0.3395,0.1280,0.1141,3.3516,0.0925
1,Flat 84 + RL,0.3418,0.1241,0.1112,3.2505,0.1132



Best RL epoch: 1
RL improved the selected validation score.

Original V2 Flat baseline: 31.60%
Test partition accessed: NO
Experiment complete.
